# Análise de detecção — baseline original, baseline ajustado, PCA e nível sustentado

Quatro abordagens passam pelo mesmo pipeline, pela mesma pontuação de alertas e pelos mesmos rótulos:

| Abordagem | O que é |
|---|---|
| **Baseline original** | O código inicial, exatamente como veio: norma da velocidade de todas as leituras (inclusive com a máquina parada), média e desvio-padrão, janela marcada quando ≥ 20% das leituras têm \|z\| > 3, e um motor de alertas que trava após o primeiro alerta. |
| **Baseline ajustado** | A mesma regra com poucas correções: só leituras com a máquina rodando, normas da velocidade *e* da aceleração, só desvios para cima, limite de 4,5 desvios-padrão e o motor de alertas por eventos que conta janelas (`classes/window_alert_engine.py`). |
| **PCA (T² + SPE)** | `classes/pca_model.py`: nível da janela nos seis canais em log, comparado com o baseline do ativo, só leituras com a máquina rodando, com o mesmo motor que conta janelas. |
| **Nível sustentado (final)** | `classes/anomaly_model.py` e `classes/alert_engine.py`, o que o `main.py` roda: normas da velocidade e da aceleração, nível sustentado nas últimas 3 h dividido pelo percentil 99 do treino, janela anormal a partir de 1,5×, e um motor de alertas que mede o tempo pelos timestamps (alerta no início do evento, re-alerta quando a severidade dobra, rearme após 24 h normais). |

Seções:

1. **O pipeline perde dados depois de um buraco** — um defeito no código fixo do janelamento e como contorná-lo.
2. **As abordagens** — implementações usadas no resto do notebook.
3. **Placar** — a métrica do desafio para cada abordagem.
4. **Do baseline original ao ajustado, uma mudança por vez.**
5. **Monitoramento com as regras do baseline ajustado** — todos os cenários, períodos de treino e de teste.
6. **Linha do tempo dos alertas** — os alertas das quatro abordagens lado a lado.
7. **Severidade e contribuições** — severidade da janela e contribuição de cada variável ao longo do tempo.
8. **As contribuições apontam a variável certa?**
9. **Como re-alertar** — nível, CUSUM ou tendência.
10. **Sensibilidade** — hiperparâmetros do PCA e janela pontuada.
11. **O que foi testado e não ajudou.**
12. **Features da janela na regra do baseline** — RMS, pico, fator de crista, curtose.
13. **O limite de 4,5 desvios-padrão é adequado?**
14. **Limiar dinâmico** — média móvel + desvio-padrão móvel.
15. **Erros que sobram e outras abordagens.**
16. **Abordagem final: nível sustentado contra o baseline do ativo** — por que, placar, ablação, sensibilidade, validação fora da amostra, os três requisitos e os erros.
17. **Que métrica olhar** — a métrica oficial e o que ela não mede.

As seções 4 a 15 estudam o baseline ajustado e o PCA; foi o que elas mostraram que levou à abordagem final da seção 16. O baseline original e o ajustado existem só neste notebook; o PCA, o motor que conta janelas e a abordagem final estão em `classes/`.

In [ ]:
import os
import sys
from collections import deque
from datetime import datetime, timedelta, timezone
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from classes.alert_engine import AlertEngine, load_alert_params
from classes.anomaly_model import AnomalyModel, load_model_params
from classes.data_pipeline import AnomalyPipeline, load_pipeline_params
from classes.pca_model import PcaAnomalyModel, _PcaDetector, load_pca_params
from classes.window_alert_engine import WindowCountAlertEngine, load_window_alert_params
from classes.interface import AlertDecision, FEATURE_NAMES, PipelineParams, PredictOutput, TimeSeries
from utils.data_loading import load_all_scenarios
from utils.utils import aggregate_results, evaluate

pd.set_option("display.max_rows", 100, "display.width", 200, "display.precision", 2)

# Abordagem final, a que o main.py roda: classes/anomaly_model.py e classes/alert_engine.py.
MODEL_PARAMS = load_model_params()
ALERT_PARAMS = load_alert_params()
PIPELINE_PARAMS = load_pipeline_params()
# Abordagens anteriores, mantidas para comparação: classes/pca_model.py e classes/window_alert_engine.py.
PCA_PARAMS = load_pca_params()
WINDOW_ALERT_PARAMS = load_window_alert_params()

APPROACHES = {"original": "Baseline original", "adjusted": "Baseline ajustado", "pca": "PCA (T² + SPE)",
              "sustained": "Nível sustentado (final)"}
APPROACH_COLORS = {"original": "#8a8985", "adjusted": "#eb6834", "pca": "#2a78d6", "sustained": "#1baf7a"}
NORM_LABELS = {"velocity": "velocidade", "acceleration": "aceleração"}
NORM_COLORS = {"velocity": "#2a78d6", "acceleration": "#1baf7a"}
METRIC_LABELS = {"precision": "Precisão", "recall": "Recall", "f1": "F1"}
METRIC_COLORS = {"precision": "#2a78d6", "recall": "#eb6834", "f1": "#1baf7a"}
ALERT_KINDS = {"onset": "novo evento", "escalation": "escalada"}
INCIDENT_FILL = "rgba(208, 59, 59, 0.18)"
ABOVE_LIMIT = "#d03b3b"
INK, NEUTRAL = "#0b0b0b", "#8a8985"
SEQUENTIAL = [[0.0, "#f0efec"], [0.25, "#9ec5f4"], [0.6, "#2a78d6"], [1.0, "#0d366b"]]
# Marcas com números inteiros nos eixos log (o padrão do plotly rotula só o primeiro dígito das marcas menores).
LOG_TICKS = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
EPS = 1e-12


def style(fig: go.Figure, title: str, height: int = 420, subtitle: str | None = None,
          legend: bool = True) -> go.Figure:
    # Visual comum: fundo branco, grade discreta. Título, subtítulo e legenda ficam empilhados em
    # faixas de pixels fixas acima da área do gráfico (coordenadas do container), então nunca se
    # sobrepõem entre si, aos títulos dos subgráficos ou aos dados, qualquer que seja a altura.
    title_px = 12
    legend_px = title_px + (52 if subtitle else 32)
    top_px = legend_px + (40 if legend else 8) + 16
    fig.update_layout(
        title=dict(
            text=title, x=0.01, xanchor="left", yref="container", y=1 - title_px / height, yanchor="top",
            font=dict(size=16), subtitle=dict(text=subtitle or "", font=dict(size=12, color="#52514e")),
        ),
        template="plotly_white",
        height=height,
        margin=dict(t=top_px, b=60, l=70, r=30),
        showlegend=legend,
        legend=dict(orientation="h", yref="container", y=1 - legend_px / height, yanchor="top",
                    x=0, xanchor="left", font=dict(size=12)),
        hovermode="x unified",
    )
    return fig


print("Modelo final:", MODEL_PARAMS.model_dump())
print("Alertas (motor final):", ALERT_PARAMS.model_dump())
print("Pipeline:", PIPELINE_PARAMS.model_dump())
print("PCA:", PCA_PARAMS.model_dump())
print("Alertas (motor por contagem de janelas):", WINDOW_ALERT_PARAMS.model_dump())

## Rodando o pipeline passo a passo

`AnomalyPipeline.predict` fatia o período de predição em janelas, pontua cada uma e entrega o resultado ao motor de alertas. As funções abaixo fazem esses três passos separadamente, para que os scores das janelas possam ser guardados, plotados e repassados por regras de alerta diferentes sem reajustar nada.

Dois janelamentos são usados no notebook inteiro (a seção 1 explica por quê):

- **`original`** — o pipeline como configurado em `hyperparameters/pipeline_hyperparams.yaml` (janelas de 2 h, passo de 1 h);
- **`gap_proof`** (janelas corrigidas) — janelas de 12 h com sobreposição de 11 h (o passo continua de 1 h), das quais o modelo pontua só as últimas `window_hours` (2 h).

In [ ]:
SCENARIOS = dict(enumerate(load_all_scenarios(), start=1))  # id -> (treino, predição, incidentes)
ORIGINAL_PIPELINE_PARAMS = PipelineParams(model_window_size_hours=2, window_overlap_hours=1)  # como veio
GAP_PROOF_PARAMS = PipelineParams(model_window_size_hours=12, window_overlap_hours=11)
WINDOWING_LABELS = {"original": "Janelas de 2 h (como veio)", "gap_proof": "Janelas corrigidas",
                    "default": "Janelas do YAML (12 h, passo de 30 min)"}


def slice_windows(pipeline_params: PipelineParams) -> dict:
    slicer = AnomalyPipeline(PcaAnomalyModel(params=PCA_PARAMS), pipeline_params=pipeline_params)._windowing_ts
    return {i: slicer(prediction) for i, (_, prediction, _) in SCENARIOS.items()}


WINDOWS = {
    "original": slice_windows(ORIGINAL_PIPELINE_PARAMS),
    "gap_proof": slice_windows(GAP_PROOF_PARAMS),
    "default": slice_windows(PIPELINE_PARAMS),  # o que o main.py usa, em hyperparameters/pipeline_hyperparams.yaml
}


def scoring_span(window: TimeSeries, hours: float = PCA_PARAMS.window_hours) -> TimeSeries:
    # As últimas `hours` de uma janela do pipeline: o que o modelo julga quando a janela é corrigida.
    last = window.data[-1].timestamp
    return TimeSeries(data=[p for p in window.data if p.timestamp > last - timedelta(hours=hours)])


def valid_incidents(scenario_id: int) -> list:
    return [inc for inc in SCENARIOS[scenario_id][2] if inc.start < inc.end]


def raise_alerts(outputs: dict, make_engine=lambda: WindowCountAlertEngine(params=WINDOW_ALERT_PARAMS)) -> dict:
    # Repassa as saídas das janelas de cada cenário por um motor de alertas novo.
    decisions = {}
    for scenario_id, scenario_outputs in outputs.items():
        engine = make_engine()
        decisions[scenario_id] = [engine.predict(output) for output in scenario_outputs]
    return decisions


def scenario_metrics(decisions: dict) -> pd.DataFrame:
    rows = {i: evaluate(SCENARIOS[i][2], decisions[i]) for i in decisions}
    table = pd.DataFrame(rows).T[["n_true_incidents", "n_predicted_alerts", "TP", "FP", "FN"]].astype(int)
    table = table.rename(columns={"n_true_incidents": "incidentes", "n_predicted_alerts": "alertas"})
    table["escaladas"] = [sum(d.kind == "escalation" for d in decisions[i]) for i in decisions]
    return table.rename_axis("cenário")


def global_metrics(decisions: dict) -> dict:
    summary = aggregate_results({i: evaluate(SCENARIOS[i][2], decisions[i]) for i in decisions})
    return {
        "TP": summary["TP_total"], "FP": summary["FP_total"], "FN": summary["FN_total"],
        "precision": summary["precision_global"], "recall": summary["recall_global"], "f1": summary["f1_global"],
    }

## 1. O pipeline perde dados depois de um buraco

`AnomalyPipeline._windowing_ts` (em `classes/data_pipeline.py`, que não pode ser modificado) para na **primeira janela vazia**. Com janelas de 2 h, qualquer trecho de mais de 2 h sem leituras encerra a predição daquele cenário, e tudo o que vem depois nunca é pontuado.

A tabela lista os cenários em que isso acontece, com as horas e os incidentes rotulados que ficam de fora.

In [ ]:
def coverage_row(scenario_id: int) -> dict:
    prediction = SCENARIOS[scenario_id][1]
    timestamps = pd.Series([p.timestamp for p in prediction.data])
    end = timestamps.iloc[-1]
    row = {"cenário": scenario_id, "maior_buraco_h": timestamps.diff().max().total_seconds() / 3600}
    for key in WINDOWS:
        last_scored = WINDOWS[key][scenario_id][-1].data[-1].timestamp
        row[f"horas_sem_pontuação ({key})"] = (end - last_scored).total_seconds() / 3600
        row[f"leituras_sem_pontuação ({key})"] = int((timestamps > last_scored).sum())
        row[f"incidentes_perdidos ({key})"] = sum(inc.start > last_scored for inc in SCENARIOS[scenario_id][2])
    first_incident = min((inc.start for inc in SCENARIOS[scenario_id][2]), default=None)
    row["horas_até_o_1º_incidente"] = (
        (first_incident - timestamps.iloc[0]).total_seconds() / 3600 if first_incident else np.nan)
    return row


coverage = pd.DataFrame([coverage_row(i) for i in SCENARIOS]).set_index("cenário")
display(coverage[coverage["horas_sem_pontuação (original)"] > 1].round(1))
print(f"Maior buraco em um período de predição: {coverage['maior_buraco_h'].max():.1f} h "
      f"(cenário {coverage['maior_buraco_h'].idxmax()})")
print(f"Incidente mais cedo: {coverage['horas_até_o_1º_incidente'].min():.0f} h após o início do seu período de predição")

**Contorno, dentro dos arquivos que podem ser alterados.** Para nenhuma janela sair vazia, basta a janela do pipeline ser mais longa que o maior buraco. Com **janelas de 12 h e sobreposição de 11 h**:

- o passo continua de 1 h, então uma decisão continua sendo tomada a cada hora;
- 12 h é mais que o maior buraco (10,1 h, cenário 28);
- o modelo pontua só as **últimas 2 h** de cada janela (`scoring_span`), o mesmo intervalo em que o baseline é aprendido, então cada decisão continua julgando as duas horas mais recentes.

O custo é que as primeiras ~10 h de cada período de predição não produzem decisão, porque nenhuma janela terminou ali ainda. O incidente mais cedo começa 64 h depois do início do seu período de predição, então nada se perde.

Em `classes/` o recorte do trecho final está no início de `PcaAnomalyModel.predict` (`classes/pca_model.py`). A abordagem final (seção 16) usa a mesma ideia com outra escala: janelas de 12 h com passo de 30 min (`hyperparameters/pipeline_hyperparams.yaml`, o janelamento `default` abaixo), das quais o modelo julga só as últimas 3 h. Daqui em diante, o baseline original aparece nos dois janelamentos de passo de 1 h, o ajustado e o PCA usam as janelas corrigidas, e a abordagem final usa as janelas do YAML, como o `main.py`.

## 2. As três abordagens

### Baseline original

O `AnomalyModel` e o `AlertEngine` do código inicial, reproduzidos **sem alterações**. Ficam como o ponto de referência contra o qual as outras duas abordagens são medidas.

In [ ]:
class OriginalBaseline:
    """O AnomalyModel do código inicial, sem alterações.

    Uma feature por leitura (a norma dos três eixos da velocidade, inclusive leituras com a
    máquina parada), sua média e seu desvio-padrão no período de treino, e uma janela marcada
    quando pelo menos `window_anomaly_ratio` das leituras estão a mais de `z_threshold` desvios,
    em qualquer direção.
    """

    def __init__(self, z_threshold: int = 3, window_anomaly_ratio: float = 0.2):
        self.z_threshold = z_threshold
        self.window_anomaly_ratio = window_anomaly_ratio
        self.mean, self.std = 0.0, 1.0

    @staticmethod
    def _featuring(samples: TimeSeries) -> np.ndarray:
        ordered = sorted(samples.data, key=lambda p: p.timestamp)
        velocity = np.array([(p.vel_x, p.vel_y, p.vel_z) for p in ordered], dtype=float)
        return np.linalg.norm(velocity, axis=1)

    def fit(self, fitting_samples: TimeSeries) -> None:
        X = self._featuring(fitting_samples)
        self.mean, self.std = round(X.mean(), 3), round(X.std(), 3)

    def predict(self, samples: TimeSeries) -> PredictOutput:
        z = np.abs((self._featuring(samples) - self.mean) / self.std)
        deviation_ratio = float(np.sum(z > self.z_threshold) / len(z))
        return PredictOutput(
            anomaly_status=deviation_ratio >= self.window_anomaly_ratio,
            timestamp=samples.data[-1].timestamp,
        )


class LatchEngine:
    """O AlertEngine do código inicial: a primeira janela anormal gera o único alerta."""

    def __init__(self):
        self.locked = False

    def predict(self, prediction: PredictOutput) -> AlertDecision:
        if self.locked:
            return AlertDecision(alert=False, timestamp=prediction.timestamp,
                                 message="System already entered abnormal state earlier.")
        if prediction.anomaly_status:
            self.locked = True
            return AlertDecision(alert=True, timestamp=prediction.timestamp,
                                 message="Abnormal vibration detected.", kind="onset")
        return AlertDecision(alert=False, timestamp=prediction.timestamp,
                             message="No persistent abnormal vibration.")

### Baseline ajustado

A regra original é mantida: um z-score por leitura, e a janela é anormal quando pelo menos 20% das leituras passam de um limite. Quatro coisas mudam, cada uma por um motivo medido na seção 4:

| Mudança | Por quê |
|---|---|
| **Motor de alertas por eventos** (`classes/alert_engine.py`) no lugar da trava | A trava permite um alerta por cenário para sempre, o que quebra os requisitos 2 e 3 e limita o recall a um incidente por cenário. O motor confirma um evento em 3 janelas, fica em silêncio enquanto ele dura, fecha o evento após 24 janelas normais e re-alerta quando a severidade dobra. |
| **Só leituras com a máquina rodando** | Leituras com a máquina parada ficam no piso de ruído do sensor. Mantidas no treino, puxam a média para baixo e inflam o desvio numa proporção que depende de quanto tempo cada máquina ficou parada (de 0% a 74% do tempo). |
| **Normas da velocidade e da aceleração** | A velocidade sozinha ignora o conteúdo de alta frequência (rolamentos, engrenagens) que aparece na aceleração. A janela é anormal se qualquer uma das normas violar a regra. |
| **Só desvios para cima, limite de 4,5 desvios-padrão** | O requisito é vibração *alta*, então uma máquina mais silenciosa não é anomalia. Sem as leituras paradas a distribuição de referência fica mais estreita, e 3 desvios disparam com a variação comum; 4,5 é o início de um platô estável (seção 4). |

A `severity` é o percentil 80 do z da janela (o nível que 20% das leituras ultrapassam) dividido pelo limite, então cruza 1 exatamente quando a regra dispara, e o motor pode comparar níveis para decidir um re-alerta. A contribuição de cada canal é o z do próprio canal nesse percentil, truncado em zero.

Os argumentos nomeados existem só para refazer os passos a partir da regra original na seção 4; os valores padrão são o baseline ajustado.

In [ ]:
VELOCITY, ACCELERATION = list(FEATURE_NAMES[:3]), list(FEATURE_NAMES[3:])


class AdjustedBaseline:
    """A regra original de z-score por leitura, com as correções listadas acima."""

    def __init__(self, z_threshold: float = 4.5, window_anomaly_ratio: float = PCA_PARAMS.window_anomaly_ratio,
                 norms: tuple = ("velocity", "acceleration"), running_only: bool = True,
                 upward_only: bool = True, min_running_samples: int = PCA_PARAMS.min_running_samples):
        self.z_threshold = z_threshold
        self.window_anomaly_ratio = window_anomaly_ratio
        self.norms = norms
        self.running_only = running_only
        self.upward_only = upward_only
        self.min_samples = min_running_samples if running_only else 1

    def _readings(self, samples: TimeSeries) -> pd.DataFrame:
        points = [p for p in samples.data if p.uptime or not self.running_only]
        return pd.DataFrame(
            [[getattr(p, name) for name in FEATURE_NAMES] for p in points],
            index=pd.DatetimeIndex([p.timestamp for p in points]), columns=list(FEATURE_NAMES), dtype=float,
        ).sort_index()

    def _norms(self, channels: pd.DataFrame) -> pd.DataFrame:
        groups = {"velocity": VELOCITY, "acceleration": ACCELERATION}
        return pd.DataFrame({name: np.linalg.norm(channels[groups[name]].to_numpy(), axis=1) for name in self.norms},
                            index=channels.index)

    def fit(self, fitting_samples: TimeSeries) -> "AdjustedBaseline":
        channels = self._readings(fitting_samples)
        norms = self._norms(channels)
        self.center, self.scale = norms.mean(), norms.std(ddof=0).clip(lower=EPS)
        self.channel_center, self.channel_scale = channels.mean(), channels.std(ddof=0).clip(lower=EPS)
        return self

    def _z(self, channels: pd.DataFrame) -> pd.DataFrame:
        z = (self._norms(channels) - self.center) / self.scale
        return z if self.upward_only else z.abs()

    def reading_z(self, samples: TimeSeries) -> pd.DataFrame:
        """z de cada norma para cada leitura que a regra olha (usado nos gráficos)."""
        return self._z(self._readings(samples))

    def share_above(self, samples: TimeSeries) -> pd.Series | None:
        """Fração das leituras acima do limite, por norma; None quando há poucas leituras."""
        channels = self._readings(samples)
        if len(channels) < self.min_samples:
            return None
        return (self._z(channels) > self.z_threshold).mean()

    def predict(self, samples: TimeSeries) -> PredictOutput:
        timestamp = samples.data[-1].timestamp
        channels = self._readings(samples)
        if len(channels) < self.min_samples:
            # Máquina (quase toda) parada: sem evidência para nenhum lado, o motor mantém o estado.
            return PredictOutput(anomaly_status=False, timestamp=timestamp, n_valid=len(channels))

        z = self._z(channels)
        share = (z > self.z_threshold).mean()
        level = z.quantile(1 - self.window_anomaly_ratio)
        channel_z = ((channels - self.channel_center) / self.channel_scale).quantile(1 - self.window_anomaly_ratio)
        return PredictOutput(
            anomaly_status=bool((share >= self.window_anomaly_ratio).any()),
            timestamp=timestamp,
            severity=max(float(level.max()) / self.z_threshold, 1e-6),
            contributions=PcaAnomalyModel._rank_contributions(channel_z.clip(lower=0).to_numpy()),
            n_valid=len(channels),
        )

### Rodando as quatro abordagens

Cada abordagem é ajustada uma vez por cenário. O baseline original, o ajustado e o PCA são pontuados nos dois janelamentos de passo de 1 h: o original usa a própria trava, e o ajustado e o PCA usam o `WindowCountAlertEngine` com `hyperparameters/window_alert_hyperparams.yaml`. A abordagem final usa o `AlertEngine` com `hyperparameters/alert_hyperparams.yaml` e as janelas do YAML, exatamente como o `main.py`.

`MAIN_WINDOWING` define a configuração com que cada abordagem aparece daqui em diante: o baseline original exatamente como veio, o baseline ajustado e o PCA nas janelas corrigidas, e a abordagem final nas janelas do YAML.

In [ ]:
MODEL_FACTORIES = {
    "original": OriginalBaseline,
    "adjusted": AdjustedBaseline,
    "pca": lambda: PcaAnomalyModel(params=PCA_PARAMS),
    "sustained": lambda: AnomalyModel(params=MODEL_PARAMS),
}
ENGINE_FACTORIES = {
    "original": LatchEngine,
    "adjusted": lambda: WindowCountAlertEngine(params=WINDOW_ALERT_PARAMS),
    "pca": lambda: WindowCountAlertEngine(params=WINDOW_ALERT_PARAMS),
    "sustained": lambda: AlertEngine(params=ALERT_PARAMS),
}
MAIN_WINDOWING = {"original": "original", "adjusted": "gap_proof", "pca": "gap_proof", "sustained": "default"}
# As três primeiras abordagens rodam nos dois janelamentos de passo de 1 h; a final, no janelamento do YAML.
RUNS = [(name, key) for name in ("original", "adjusted", "pca") for key in ("original", "gap_proof")]
RUNS.append(("sustained", "default"))


def fit_all(make_model) -> dict:
    models = {}
    for scenario_id, (fit, _, _) in SCENARIOS.items():
        model = make_model()
        model.fit(fit)
        models[scenario_id] = model
    return models


def score_all(models: dict, windowing: str, span_hours: float = PCA_PARAMS.window_hours,
              windows: dict | None = None) -> dict:
    # Nas janelas corrigidas, as abordagens anteriores julgam só as últimas `span_hours`; o modelo final
    # recorta o próprio horizonte dentro de `predict`.
    windows = windows or WINDOWS[windowing]
    cut = (lambda w: scoring_span(w, span_hours)) if windowing == "gap_proof" else (lambda w: w)
    return {i: [models[i].predict(cut(window)) for window in windows[i]] for i in SCENARIOS}


MODELS = {name: fit_all(MODEL_FACTORIES[name]) for name in APPROACHES}
OUTPUTS = {(name, key): score_all(MODELS[name], key) for name, key in RUNS}
DECISIONS = {(name, key): raise_alerts(OUTPUTS[(name, key)], ENGINE_FACTORIES[name]) for name, key in OUTPUTS}


def main_outputs(name: str) -> dict:
    return OUTPUTS[(name, MAIN_WINDOWING[name])]


def main_decisions(name: str) -> dict:
    return DECISIONS[(name, MAIN_WINDOWING[name])]


## 3. Placar

Um alerta é verdadeiro positivo quando seu timestamp cai dentro de uma janela rotulada; um incidente é encontrado quando pelo menos um alerta cai nele; todo alerta fora de uma janela é falso positivo. Um incidente (cenário 7) termina antes de começar e nunca pode ser encontrado, então o melhor recall possível é 28/29.

In [ ]:
PREVIOUS = {name: APPROACHES[name] for name in ("original", "adjusted", "pca")}
scoreboard = pd.concat({
    WINDOWING_LABELS[key]: pd.DataFrame({label: global_metrics(DECISIONS[(name, key)])
                                         for name, label in PREVIOUS.items()}).T
    for key in ("original", "gap_proof")
}, axis=1)
for key in ("original", "gap_proof"):
    scoreboard[[(WINDOWING_LABELS[key], c) for c in ("TP", "FP", "FN")]] = (
        scoreboard[[(WINDOWING_LABELS[key], c) for c in ("TP", "FP", "FN")]].astype(int))
display(scoreboard.round(3))

main = pd.DataFrame({label: global_metrics(main_decisions(name)) for name, label in APPROACHES.items()}).T
main[["TP", "FP", "FN"]] = main[["TP", "FP", "FN"]].astype(int)
display(main.round(3))

fig = go.Figure()
for metric, color in METRIC_COLORS.items():
    fig.add_bar(
        x=main.index, y=main[metric], name=METRIC_LABELS[metric],
        marker=dict(color=color, line=dict(color="white", width=2)),
        text=main[metric].map("{:.2f}".format), textposition="outside", cliponaxis=False,
        hovertemplate="%{y:.2f}<extra>" + METRIC_LABELS[metric] + "</extra>",
    )
style(fig, "Precisão, recall e F1 globais",
      subtitle="Baseline original como veio; ajustado e PCA nas janelas corrigidas; final nas janelas do YAML")
fig.update_layout(barmode="group", bargap=0.35, hovermode="closest")
fig.update_yaxes(range=[0, 1.05], title_text="Valor")
fig.show()


In [ ]:
per_scenario = pd.concat({label: scenario_metrics(main_decisions(name)) for name, label in APPROACHES.items()}, axis=1)
per_scenario.insert(0, "incidentes", per_scenario[(APPROACHES["pca"], "incidentes")])
per_scenario = per_scenario.drop(columns=["incidentes", "escaladas"], level=1)
display(per_scenario)

scored = [(a, b) for i in SCENARIOS for a, b in zip(main_outputs("adjusted")[i], main_outputs("pca")[i])
          if a.severity is not None and b.severity is not None]
agree = np.mean([a.anomaly_status == b.anomaly_status for a, b in scored])
print(f"Janelas pontuadas pelas duas abordagens: {len(scored)}. "
      f"O baseline ajustado e o PCA concordam em normal/anormal em {agree:.1%} delas.")

## 4. Do baseline original ao ajustado, uma mudança por vez

Cada linha acrescenta uma mudança à linha de cima. Todas usam as janelas de 2 h do pipeline como veio, exceto a última, que passa para as janelas corrigidas. A segunda tabela mostra como o resultado varia com o limite `z_threshold`, com e sem as leituras paradas.

In [ ]:
def run_variant(make_model, make_engine, windowing: str) -> dict:
    outputs = score_all(fit_all(make_model), windowing)
    return global_metrics(raise_alerts(outputs, make_engine))


event_engine = lambda: WindowCountAlertEngine(params=WINDOW_ALERT_PARAMS)
LADDER = [
    ("Baseline original", OriginalBaseline, LatchEngine, "original"),
    ("+ motor de alertas por eventos", lambda: AdjustedBaseline(
        z_threshold=3, norms=("velocity",), running_only=False, upward_only=False), event_engine, "original"),
    ("+ só desvios para cima", lambda: AdjustedBaseline(
        z_threshold=3, norms=("velocity",), running_only=False), event_engine, "original"),
    ("+ norma da aceleração", lambda: AdjustedBaseline(z_threshold=3, running_only=False), event_engine, "original"),
    ("+ só leituras rodando (limite 3)", lambda: AdjustedBaseline(z_threshold=3), event_engine, "original"),
    ("+ limite 4,5", AdjustedBaseline, event_engine, "original"),
    ("+ janelas corrigidas (baseline ajustado)", AdjustedBaseline, event_engine, "gap_proof"),
]
ladder = pd.DataFrame({step: run_variant(*args) for step, *args in LADDER}).T
ladder[["TP", "FP", "FN"]] = ladder[["TP", "FP", "FN"]].astype(int)
display(ladder.round(3))

In [ ]:
limit_sweep = pd.DataFrame({
    (("todas as leituras" if not running_only else "só leituras rodando"), k): run_variant(
        lambda k=k, running_only=running_only: AdjustedBaseline(z_threshold=k, running_only=running_only),
        event_engine, "gap_proof")
    for running_only in (False, True) for k in (3, 3.5, 4, 4.5, 5)
}).T.rename_axis(["leituras", "z_threshold"])
limit_sweep[["TP", "FP", "FN"]] = limit_sweep[["TP", "FP", "FN"]].astype(int)
display(limit_sweep.round(3))

**Lendo as duas tabelas.**

- **O motor de alertas é o maior ganho isolado.** Trocar a trava pelo motor por eventos leva o F1 de 0,21 a 0,49 sem mexer no detector. A trava permite um alerta por cenário, então nunca encontra o segundo incidente de um cenário, e nos cenários sem incidente a primeira janela ruidosa vira falso positivo (15, 23, 24 e 25 ganham um cada).
- **Só desvios para cima** remove 3 falsos positivos (11 → 8) sem custo de recall: eram janelas em que a máquina rodava mais silenciosa que o normal.
- **A norma da aceleração** encontra mais 3 incidentes (13 → 16) que aparecem na aceleração mas não na velocidade, ao custo de 6 falsos positivos a mais.
- **Descartar as leituras paradas é o certo, mas exige um novo limite.** Sem as leituras no piso de ruído a média sobe e o desvio-padrão encolhe, então 3 desvios viram um limite bem mais apertado e os falsos positivos saltam de 14 para 25. Com 4,5 desvios o resultado volta a 14 / 9 / 15, com a melhor precisão da tabela (0,61).
- **As janelas corrigidas** pontuam os dados que o pipeline descartava: 2 incidentes a mais por 3 falsos positivos a mais (F1 0,54 → 0,56).
- **A tabela de limites explica por que as leituras paradas precisam sair.** Com elas, o F1 oscila de 0,41 a 0,60 conforme o limite vai de 3 a 5: o desvio é inflado por quanto tempo cada máquina por acaso ficou parada (0% a 74% do tempo), então o limite efetivo muda de ativo para ativo por acidente, e o 0,60 em 4,0 é um ponto de sorte, não uma configuração confiável. Só com leituras rodando, o recall fica em 16 incidentes de 3,5 a 5 enquanto os falsos positivos caem até 4,5, e a partir daí nada muda — é o platô que fixa o limite em 4,5.

## 5. Monitoramento com as regras do baseline ajustado

Uma figura por cenário, com o período de treino (fit) à esquerda e o período de teste (predição) à direita:

- **Linhas 1 e 2** — z da norma da velocidade e da norma da aceleração em cada leitura com a máquina rodando, em desvios-padrão do período de treino. A linha tracejada é o limite (4,5); as leituras acima dele aparecem em vermelho. Valores além do eixo são desenhados na borda superior, com o valor real no hover.
- **Linha 3** — fração das leituras acima do limite nas últimas 2 h, na pior das duas normas. A janela é anormal quando essa fração chega a 20% (linha tracejada); três janelas anormais seguidas abrem um evento (▲), e a severidade dobrar com o evento aberto gera um re-alerta (◆). No lado do treino, a mesma fração é calculada numa janela móvel de 2 h, para mostrar com que frequência a regra dispararia em dados saudáveis.
- **Faixas vermelhas** — incidentes rotulados.

In [ ]:
def period_days(ts: TimeSeries) -> float:
    return (ts.data[-1].timestamp - ts.data[0].timestamp).total_seconds() / 86400


def plot_adjusted_monitoring(scenario_id: int) -> None:
    model = MODELS["adjusted"][scenario_id]
    fit, prediction, _ = SCENARIOS[scenario_id]
    k, ratio = model.z_threshold, model.window_anomaly_ratio
    periods = {"fit": fit, "pred": prediction}

    # z de cada leitura rodando, reindexado em todos os timestamps para a linha quebrar quando a máquina para.
    z = {key: model.reading_z(ts).reindex(pd.DatetimeIndex([p.timestamp for p in ts.data])) for key, ts in periods.items()}
    stacked = pd.concat(z.values()).stack()
    cap = max(2 * k, min(stacked.max(), 2 * stacked.quantile(0.999))) * 1.05
    floor = min(-3.0, stacked.quantile(0.001) * 1.05)

    fit_share = ((model.reading_z(fit) > k).rolling(f"{PCA_PARAMS.window_hours}h", min_periods=model.min_samples)
                 .mean().max(axis=1).reindex(z["fit"].index))
    # Uma linha por janela corrigida. Janelas que terminam na mesma leitura (dentro de um buraco)
    # repetem o timestamp, então a tabela mantém um índice posicional.
    shares = [model.share_above(scoring_span(w)) for w in WINDOWS["gap_proof"][scenario_id]]
    pred_windows = pd.DataFrame({
        "end": [w.data[-1].timestamp for w in WINDOWS["gap_proof"][scenario_id]],
        "share": [s.max() if s is not None else np.nan for s in shares],
        "alert": [d.kind for d in main_decisions("adjusted")[scenario_id]],
    })
    pred_share = pred_windows.set_index("end")["share"]

    days = np.array([period_days(fit), period_days(prediction)])
    widths = np.maximum(days, 0.35 * days.sum())
    fig = make_subplots(
        rows=3, cols=2, shared_xaxes="columns", shared_yaxes="rows", row_heights=[0.36, 0.36, 0.28],
        column_widths=list(widths / widths.sum()), vertical_spacing=0.06, horizontal_spacing=0.02,
        column_titles=[f"<b>Período de treino (fit)</b><br>{fit.data[0].timestamp:%d/%m} – {fit.data[-1].timestamp:%d/%m/%Y}",
                       f"<b>Período de teste (predição)</b><br>{prediction.data[0].timestamp:%d/%m} – {prediction.data[-1].timestamp:%d/%m/%Y}"],
    )
    for col, key in enumerate(periods, start=1):
        for row, norm in enumerate(model.norms, start=1):
            series = z[key][norm]
            fig.add_trace(go.Scatter(
                x=series.index, y=series.clip(upper=cap * 0.97), mode="lines", connectgaps=False,
                name=f"z da {NORM_LABELS[norm]}", legendgroup=norm, showlegend=col == 1,
                line=dict(color=NORM_COLORS[norm], width=1), customdata=series,
                hovertemplate="%{customdata:.2f}<extra>z da " + NORM_LABELS[norm] + "</extra>",
            ), row=row, col=col)
            above = series[series > k]
            fig.add_trace(go.Scatter(
                x=above.index, y=above.clip(upper=cap * 0.97), mode="markers", customdata=above,
                name="Acima do limite", legendgroup="above", showlegend=False,
                marker=dict(color=ABOVE_LIMIT, size=5, line=dict(color="white", width=0.5)),
                hovertemplate="%{customdata:.2f}<extra>acima do limite</extra>",
            ), row=row, col=col)
            fig.add_hline(y=k, line=dict(color=INK, width=1, dash="dash"), row=row, col=col)
        share = fit_share if key == "fit" else pred_share
        fig.add_trace(go.Scatter(
            x=share.index, y=share, mode="lines", connectgaps=False, name="Fração acima do limite",
            legendgroup="share", showlegend=col == 1, line=dict(color=APPROACH_COLORS["adjusted"], width=1.5, shape="hv"),
            hovertemplate="%{y:.0%}<extra>fração acima do limite</extra>",
        ), row=3, col=col)
        fig.add_hline(y=ratio, line=dict(color=INK, width=1, dash="dash"), row=3, col=col)

    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="Acima do limite", legendgroup="above",
                             marker=dict(color=ABOVE_LIMIT, size=7)), row=1, col=1)
    for kind, label, symbol in [("onset", "Novo evento", "triangle-up"), ("escalation", "Escalada", "diamond")]:
        when = pred_windows[pred_windows["alert"] == kind]
        fig.add_trace(go.Scatter(
            x=when["end"], y=when["share"], mode="markers", name=label,
            marker=dict(color=INK, size=11, symbol=symbol, line=dict(color="white", width=2)),
            hovertemplate="%{x|%d/%m %H:%M}<extra>" + label + "</extra>",
        ), row=3, col=2)
    for incident in valid_incidents(scenario_id):
        fig.add_vrect(x0=incident.start, x1=incident.end, fillcolor=INCIDENT_FILL, line_width=0,
                      layer="below", row="all", col=2)
    if valid_incidents(scenario_id):
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="Incidente",
                                 marker=dict(color=INCIDENT_FILL, symbol="square", size=14)), row=1, col=2)

    counts = scenario_metrics({scenario_id: main_decisions("adjusted")[scenario_id]}).iloc[0]
    style(fig, f"Cenário {scenario_id} — baseline ajustado", height=720,
          subtitle=f"{counts['incidentes']} incidente(s) rotulado(s) · {counts['alertas']} alerta(s) · "
                   f"TP {counts['TP']}, FP {counts['FP']}, FN {counts['FN']} · tracejado: limite z = {k} "
                   f"e {ratio:.0%} da janela")
    fig.update_layout(margin_t=fig.layout.margin.t + 16, hovermode="closest")
    fig.update_annotations(font_size=13)
    fig.update_yaxes(range=[floor, cap], row=1)
    fig.update_yaxes(range=[floor, cap], row=2)
    fig.update_yaxes(range=[0, 1.05], tickformat=".0%", row=3)
    fig.update_yaxes(title_text="z da norma<br>da velocidade", row=1, col=1)
    fig.update_yaxes(title_text="z da norma<br>da aceleração", row=2, col=1)
    fig.update_yaxes(title_text="Fração acima<br>do limite", row=3, col=1)
    fig.update_xaxes(hoverformat="%d/%m/%Y %H:%M")
    fig.update_xaxes(nticks=6, tickformat="%d/%m/%y", row=3)
    fig.show()

In [ ]:
for scenario_id in SCENARIOS:
    plot_adjusted_monitoring(scenario_id)

## 6. Linha do tempo dos alertas das quatro abordagens

Uma linha por cenário, com o tempo contado a partir do início do período de predição. Cada abordagem tem sua própria faixa dentro da linha (de cima para baixo: original, ajustado, PCA e final). Marcador cheio é alerta dentro de um incidente rotulado (verdadeiro positivo); vazado é alerta fora (falso positivo). ▲ abre um evento, ◆ é um re-alerta.

In [ ]:
LANES = {"original": -0.33, "adjusted": -0.11, "pca": 0.11, "sustained": 0.33}


def alert_points(name: str) -> pd.DataFrame:
    rows = []
    for scenario_id, decisions in main_decisions(name).items():
        start = SCENARIOS[scenario_id][1].data[0].timestamp
        for decision in decisions:
            if decision.alert:
                rows.append({
                    "scenario": scenario_id, "timestamp": decision.timestamp, "kind": decision.kind,
                    "days": (decision.timestamp - start).total_seconds() / 86400,
                    "inside": any(inc.start <= decision.timestamp <= inc.end for inc in valid_incidents(scenario_id)),
                })
    return pd.DataFrame(rows)


fig = go.Figure()
for scenario_id, (_, prediction, _) in SCENARIOS.items():
    start = prediction.data[0].timestamp
    fig.add_shape(type="line", x0=0, x1=period_days(prediction), y0=scenario_id, y1=scenario_id,
                  line=dict(color="#d8d7d2", width=1), layer="below")
    for incident in valid_incidents(scenario_id):
        fig.add_shape(type="rect", x0=(incident.start - start).total_seconds() / 86400,
                      x1=(incident.end - start).total_seconds() / 86400,
                      y0=scenario_id - 0.45, y1=scenario_id + 0.45, fillcolor=INCIDENT_FILL, line_width=0, layer="below")

for name, label in APPROACHES.items():
    points = alert_points(name)
    symbols = points["kind"].map({"onset": "triangle-up", "escalation": "diamond"}).fillna("circle")
    symbols = symbols.where(points["inside"], symbols + "-open")
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=label, legendgroup=name,
                             marker=dict(color=APPROACH_COLORS[name], size=10, symbol="square")))
    fig.add_trace(go.Scatter(
        x=points["days"], y=points["scenario"] + LANES[name], mode="markers", name=label,
        legendgroup=name, showlegend=False,
        marker=dict(color=APPROACH_COLORS[name], size=9, symbol=symbols, line=dict(width=1.5, color=APPROACH_COLORS[name])),
        customdata=np.stack([points["timestamp"].dt.strftime("%d/%m/%Y %H:%M"), points["kind"].map(ALERT_KINDS),
                             points["inside"].map({True: "dentro de", False: "fora de"})], axis=1),
        hovertemplate="Cenário %{y:.0f} · %{customdata[0]}<br>%{customdata[1]}, %{customdata[2]} um incidente<extra>" + label + "</extra>",
    ))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="Incidente rotulado",
                         marker=dict(color=INCIDENT_FILL, symbol="square", size=14)))

totals = " · ".join(f"{APPROACHES[n]}: TP {m['TP']}, FP {m['FP']}"
                    for n, m in ((n, global_metrics(main_decisions(n))) for n in APPROACHES))
style(fig, "Alertas das quatro abordagens", height=1200,
      subtitle=f"Cheio = dentro de um incidente rotulado, vazado = fora · {totals}")
fig.update_layout(hovermode="closest")
fig.update_yaxes(tickvals=list(SCENARIOS), ticktext=[f"Cenário {i}" for i in SCENARIOS],
                 range=[len(SCENARIOS) + 0.7, 0.3], showgrid=False, zeroline=False)
fig.update_xaxes(title_text="Dias desde o início do período de predição", rangemode="tozero")
fig.show()

## 7. Severidade e contribuições

- **Em cima** — severidade da janela no baseline ajustado, no PCA e na abordagem final (escala log). As três estão em unidades do próprio limite de alarme (a da final, nível ÷ baseline, é dividida pelo limite de 1,5), então a linha pontilhada em 1 é onde a janela fica anormal para qualquer uma delas.
- **Marcadores** — alertas da abordagem passada em `approach`: ▲ abre um evento, ◆ é um re-alerta por escalada.
- **Embaixo** — fração do desvio atribuída a cada variável por essa abordagem, só nas janelas anormais (o baseline ajustado e o PCA; a abordagem final informa só a norma que puxou a severidade, na seção 16).
- **Faixas vermelhas** — incidentes rotulados.

Todos os cenários aparecem abaixo com o PCA; `plot_detection(scenario_id, "adjusted")` mostra os alertas e as contribuições do baseline ajustado.

In [ ]:
SCORED = {"adjusted": APPROACHES["adjusted"], "pca": APPROACHES["pca"]}  # abordagens com contribuições por variável
SEVERITY_LINES = {**SCORED, "sustained": APPROACHES["sustained"]}
# A severidade do modelo final é um múltiplo do baseline; dividida pelo limite, fica na escala "limite = 1".
SEVERITY_UNIT = {"sustained": MODEL_PARAMS.severity_threshold}


def window_frame(scenario_id: int, name: str) -> pd.DataFrame:
    # Severidade, status e frações de contribuição de cada janela de um cenário.
    rows = []
    for output, decision in zip(main_outputs(name)[scenario_id], main_decisions(name)[scenario_id]):
        rows.append({
            "timestamp": output.timestamp,
            "severity": np.nan if output.severity is None else output.severity / SEVERITY_UNIT.get(name, 1.0),
            "abnormal": output.anomaly_status, "alert": decision.kind,
            **{feature: output.contributions.get(feature, np.nan) for feature in FEATURE_NAMES},
        })
    return pd.DataFrame(rows).set_index("timestamp")


def plot_detection(scenario_id: int = 3, approach: str = "pca") -> None:
    frames = {name: window_frame(scenario_id, name) for name in SEVERITY_LINES}
    chosen = frames[approach]
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, row_heights=[0.62, 0.38],
        subplot_titles=("<b>Severidade da janela</b> (limite de alarme = 1)",
                        f"<b>Contribuição de cada variável</b> — {SCORED[approach]}, janelas anormais"),
    )
    for name, label in SEVERITY_LINES.items():
        fig.add_trace(go.Scatter(
            x=frames[name].index, y=frames[name]["severity"].clip(lower=0.05), name=label, mode="lines",
            line=dict(color=APPROACH_COLORS[name], width=1.5), connectgaps=False,
            hovertemplate="%{y:.2f}<extra>" + label + "</extra>",
        ), row=1, col=1)
    fig.add_hline(y=1, line=dict(color=NEUTRAL, width=1, dash="dot"), row=1, col=1)

    for kind, label, symbol in [("onset", "Alerta: novo evento", "triangle-up"), ("escalation", "Alerta: escalada", "diamond")]:
        alerts = chosen[chosen["alert"] == kind]
        fig.add_trace(go.Scatter(
            x=alerts.index, y=alerts["severity"].clip(lower=0.05), name=label, mode="markers",
            marker=dict(color=INK, size=11, symbol=symbol, line=dict(color="white", width=2)),
            hovertemplate="%{x|%d/%m %H:%M} — severidade %{y:.2f}<extra>" + label + "</extra>",
        ), row=1, col=1)

    shares = chosen[list(FEATURE_NAMES)].where(chosen["abnormal"])
    fig.add_trace(go.Heatmap(
        x=shares.index, y=list(FEATURE_NAMES), z=shares.values.T, zmin=0, zmax=1, colorscale=SEQUENTIAL,
        colorbar=dict(title="Fração", len=0.36, y=0, yanchor="bottom", thickness=14, tickformat=".0%"),
        hovertemplate="%{x|%d/%m %H:%M} — %{y}: %{z:.0%}<extra></extra>",
    ), row=2, col=1)

    for incident in valid_incidents(scenario_id):
        fig.add_vrect(x0=incident.start, x1=incident.end, fillcolor=INCIDENT_FILL, line_width=0, layer="below")
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="Incidente rotulado",
                             marker=dict(color=INCIDENT_FILL, symbol="square", size=14)), row=1, col=1)

    counts = scenario_metrics({scenario_id: main_decisions(approach)[scenario_id]}).iloc[0]
    style(fig, f"Cenário {scenario_id} — {SCORED[approach]}", height=660,
          subtitle=f"{counts['incidentes']} incidente(s) rotulado(s) · {counts['alertas']} alerta(s) · "
                   f"TP {counts['TP']}, FP {counts['FP']}, FN {counts['FN']}")
    fig.update_layout(hovermode="closest")
    fig.update_annotations(font_size=13)
    fig.update_yaxes(type="log", tickvals=LOG_TICKS, title_text="Severidade", row=1, col=1)
    fig.update_yaxes(autorange="reversed", row=2, col=1)
    fig.update_xaxes(tickformat="%d/%m %H:%M", hoverformat="%d/%m/%Y %H:%M")
    fig.update_xaxes(title_text="Tempo (UTC)", row=2, col=1)
    fig.show()

In [ ]:
for scenario_id in SCENARIOS:
    plot_detection(scenario_id, "pca")

### Quais variáveis puxam os alertas?

Para cada alerta, a variável que o baseline ajustado e o PCA colocam em primeiro lugar.

In [ ]:
def alert_table(name: str) -> pd.DataFrame:
    rows = []
    for scenario_id in SCENARIOS:
        for output, decision in zip(main_outputs(name)[scenario_id], main_decisions(name)[scenario_id]):
            if decision.alert:
                ranked = list(output.contributions.items())
                rows.append({
                    "cenário": scenario_id, "timestamp": decision.timestamp, "tipo": ALERT_KINDS[decision.kind],
                    "dentro_do_incidente": any(inc.start <= decision.timestamp <= inc.end
                                               for inc in valid_incidents(scenario_id)),
                    "severidade": output.severity, "variável_principal": ranked[0][0], "fração_principal": ranked[0][1],
                    "mensagem": decision.message,
                })
    return pd.DataFrame(rows)


alerts = {name: alert_table(name) for name in SCORED}
pd.set_option("display.max_colwidth", 120)
display(alerts["pca"].head(8))

top = pd.DataFrame({label: alerts[name]["variável_principal"].value_counts() for name, label in SCORED.items()})
top = top.reindex(FEATURE_NAMES).fillna(0).astype(int)
fig = go.Figure()
for name, label in SCORED.items():
    fig.add_bar(x=top.index, y=top[label], name=label,
                marker=dict(color=APPROACH_COLORS[name], line=dict(color="white", width=2)),
                hovertemplate="%{y} alertas<extra>" + label + "</extra>")
style(fig, "Variável em primeiro lugar em cada alerta", subtitle="Número de alertas, todos os cenários")
fig.update_layout(barmode="group", bargap=0.3, hovermode="closest")
fig.update_yaxes(title_text="Alertas")
fig.show()

## 8. As contribuições apontam a variável certa?

Incidentes reais não dizem qual sensor está com defeito, então a ordenação é testada onde a resposta é conhecida. Para cada cenário, pegam-se as últimas 2 h do período de treino com a máquina rodando (saudáveis por construção) e multiplica-se um canal por um fator; a abordagem deveria colocar esse canal em primeiro lugar. Um segundo teste multiplica **os seis** canais de uma vez: nenhuma variável isolada é a culpada, mas a janela ainda precisa ser marcada.

In [ ]:
INJECTION_FACTORS = (1.5, 2.0, 3.0)


def healthy_window(scenario_id: int) -> TimeSeries:
    # Últimas `window_hours` do período de treino com a máquina rodando.
    running = [p for p in SCENARIOS[scenario_id][0].data if p.uptime]
    start = running[-1].timestamp - timedelta(hours=PCA_PARAMS.window_hours)
    return TimeSeries(data=[p for p in running if p.timestamp > start])


def inject(window: TimeSeries, channels: tuple, factor: float) -> TimeSeries:
    scaled = lambda p: p.model_copy(update={c: getattr(p, c) * factor for c in channels})
    return TimeSeries(data=[scaled(p) for p in window.data])


def injection_test(name: str) -> pd.DataFrame:
    rows = []
    for scenario_id in SCENARIOS:
        window = healthy_window(scenario_id)
        if len(window.data) < PCA_PARAMS.min_running_samples:
            continue
        model = MODELS[name][scenario_id]
        for factor in INJECTION_FACTORS:
            for channel in FEATURE_NAMES:
                output = model.predict(inject(window, (channel,), factor))
                ranking = list(output.contributions)
                rows.append({"abordagem": SCORED[name], "fator": factor, "teste": "um canal",
                             "sinalizado": output.anomaly_status, "injetado_em_primeiro": ranking[0] == channel,
                             "fração": output.contributions[channel]})
            output = model.predict(inject(window, FEATURE_NAMES, factor))
            rows.append({"abordagem": SCORED[name], "fator": factor, "teste": "todos os canais",
                         "sinalizado": output.anomaly_status, "injetado_em_primeiro": np.nan, "fração": np.nan})
    return pd.DataFrame(rows)


injections = pd.concat([injection_test(name) for name in SCORED])
summary = injections.groupby(["teste", "fator", "abordagem"]).agg(
    casos=("sinalizado", "size"), sinalizados=("sinalizado", "mean"),
    injetado_em_primeiro=("injetado_em_primeiro", "mean"), fração_mediana=("fração", "median"),
)
display(summary.round(2))

## 9. Como re-alertar: nível, CUSUM ou tendência?

O produto pede um alerta por evento e um novo só se a condição piorar. Três definições de "piorar" são comparadas nos **mesmos** scores de janela do PCA (janelas corrigidas) e com as **mesmas** regras para abrir e fechar um evento, então só a regra de re-alerta muda:

| Regra | Re-alerta quando… |
|---|---|
| **Nenhuma** | nunca (um alerta por evento) |
| **Nível** (padrão do motor) | as últimas janelas ficam todas acima de `escalation_ratio` × o nível do último alerta |
| **CUSUM** | o excesso acumulado do log da severidade sobre o último nível alertado cruza um limite — a forma sequencial de uma comparação de médias, que também soma pequenos aumentos sustentados |
| **Tendência** | uma reta ajustada às últimas seis janelas do log da severidade sobe mais que `escalation_ratio` |

Cada regra é uma subclasse de `AlertEngine` que sobrescreve um método, `_escalated_level`.

In [ ]:
PCA_OUTPUTS = main_outputs("pca")
LOG_RATIO = np.log(WINDOW_ALERT_PARAMS.escalation_ratio)


class NoEscalationEngine(WindowCountAlertEngine):
    def _escalated_level(self):
        return None


class CusumEngine(WindowCountAlertEngine):
    # CUSUM unilateral sobre log(severidade / nível no último alerta).
    SLACK = LOG_RATIO / 2  # metade do desvio que vale a pena detectar, a escolha usual
    LIMIT = WINDOW_ALERT_PARAMS.escalation_windows * LOG_RATIO / 2  # um desvio de exatamente LOG_RATIO dispara em escalation_windows

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cusum, self._cusum_reference = 0.0, None

    def _escalated_level(self):
        if self._cusum_reference != self._reference_level:  # novo evento ou novo degrau: recomeça
            self._cusum, self._cusum_reference = 0.0, self._reference_level
        severity = self._recent[-1]
        self._cusum = max(0.0, self._cusum + np.log(severity / self._reference_level) - self.SLACK)
        return severity if self._cusum >= self.LIMIT else None


class TrendEngine(WindowCountAlertEngine):
    # Inclinação por mínimos quadrados do log da severidade nas últimas LOOKBACK janelas anormais.
    LOOKBACK = 6

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._recent = deque(maxlen=self.LOOKBACK)

    def _escalated_level(self):
        if len(self._recent) < self.LOOKBACK:
            return None
        log_severity = np.log(self._recent)
        slope = np.polyfit(np.arange(self.LOOKBACK), log_severity, 1)[0]
        if slope * (self.LOOKBACK - 1) < LOG_RATIO:
            return None
        level = float(np.mean(list(self._recent)[-WINDOW_ALERT_PARAMS.escalation_windows:]))
        self._recent.clear()  # a mesma subida não pode disparar duas vezes
        return level


RULES = {"Nenhuma": NoEscalationEngine, "Nível": WindowCountAlertEngine, "CUSUM": CusumEngine, "Tendência": TrendEngine}


def rule_row(engine_class) -> dict:
    decisions = raise_alerts(PCA_OUTPUTS, lambda: engine_class(params=WINDOW_ALERT_PARAMS))
    re_alerts = [
        any(inc.start <= d.timestamp <= inc.end for inc in valid_incidents(i))
        for i in decisions for d in decisions[i] if d.kind == "escalation"
    ]
    return {**global_metrics(decisions), "re_alertas": len(re_alerts),
            "re_alertas_no_incidente": int(sum(re_alerts))}


rules = pd.DataFrame({name: rule_row(engine) for name, engine in RULES.items()}).T
rules[["TP", "FP", "FN", "re_alertas", "re_alertas_no_incidente"]] = rules[
    ["TP", "FP", "FN", "re_alertas", "re_alertas_no_incidente"]].astype(int)
display(rules)

In [ ]:
fig = go.Figure()
for values, name, color in [
    (rules["re_alertas_no_incidente"], "Re-alertas dentro de um incidente rotulado", "#2a78d6"),
    (rules["re_alertas"] - rules["re_alertas_no_incidente"], "Re-alertas fora de qualquer incidente rotulado", "#b9b8b3"),
]:
    fig.add_bar(x=rules.index, y=values, name=name, text=[v if v else "" for v in values],
                textposition="inside", insidetextanchor="middle", textangle=0, constraintext="none",
                marker=dict(color=color, line=dict(color="white", width=2)),
                hovertemplate="%{y}<extra>" + name + "</extra>")
style(fig, "Re-alertas gerados por cada regra", subtitle="Scores do PCA nas janelas corrigidas, todos os cenários")
fig.update_layout(barmode="stack", bargap=0.45, hovermode="closest", legend_traceorder="normal")
fig.update_xaxes(title_text="Regra de re-alerta")
fig.update_yaxes(title_text="Re-alertas")
fig.show()

### As mesmas regras em quatro perfis de manual

O placar acima muda por poucos alertas, poucos demais para separar as regras. O comportamento delas é mais fácil de julgar em perfis de severidade em que a resposta certa é conhecida. Cada perfil começa com um evento em 2x o limite (que toda regra alerta uma vez); a tabela conta os **re-alertas** que vêm depois.

| Perfil | O que faz depois do primeiro alerta | Re-alertas desejados |
|---|---|---|
| Platô | fica em 2x por 40 janelas | 0 |
| Degrau | salta de 2x para 5x e fica | 1 |
| Rampa lenta | cresce 5% por janela, de 2x a 14x | 2 (um a cada dobra) |
| Dente de serra | oscila entre 2x e 3,5x | 0 |

In [ ]:
PROFILES = {
    "Platô": [2.0] * 40,
    "Degrau": [2.0] * 15 + [5.0] * 25,
    "Rampa lenta": [2.0 * 1.05**k for k in range(40)],
    "Dente de serra": [2.0, 2.0, 2.0] + [2.0, 2.6, 3.5, 2.6] * 9 + [2.0],
}


def count_re_alerts(engine_class, severities: list) -> int:
    engine = engine_class(params=WINDOW_ALERT_PARAMS)
    start = datetime(2026, 1, 1, tzinfo=timezone.utc)
    decisions = [
        engine.predict(PredictOutput(anomaly_status=True, timestamp=start + timedelta(hours=k), severity=s))
        for k, s in enumerate(severities)
    ]
    return sum(d.kind == "escalation" for d in decisions)


profile_table = pd.DataFrame(
    {rule: {name: count_re_alerts(engine, profile) for name, profile in PROFILES.items()}
     for rule, engine in RULES.items()}
)
profile_table.insert(0, "desejado", [0, 1, 2, 0])
display(profile_table)

## 10. Sensibilidade aos principais hiperparâmetros

Cada painel varia um hiperparâmetro e mantém os outros nos valores de `hyperparameters/*.yaml` (linha pontilhada). Detector PCA, janelas corrigidas. Os rótulos usados para pontuar são os mesmos usados para escolher os valores, então essas curvas mostram o quanto a escolha é *estável*; não são uma estimativa de desempenho em ativos novos.

In [ ]:
def score_pca(model_params=PCA_PARAMS, windows: dict | None = None) -> dict:
    models = fit_all(lambda: PcaAnomalyModel(params=model_params))
    return score_all(models, "gap_proof", span_hours=model_params.window_hours, windows=windows)


def metrics_for(model_update=None, alert_update=None) -> dict:
    outputs = PCA_OUTPUTS if not model_update else score_pca(PCA_PARAMS.model_copy(update=model_update))
    alert_params = WINDOW_ALERT_PARAMS.model_copy(update=alert_update or {})
    return global_metrics(raise_alerts(outputs, lambda: WindowCountAlertEngine(params=alert_params)))


SWEEPS = {
    "threshold_margin": ("model", [1.5, 2.0, 2.5, 3.0, 4.0]),
    "window_anomaly_ratio": ("model", [0.1, 0.2, 0.3, 0.4, 0.5]),
    "onset_windows": ("alert", [1, 2, 3, 4, 6]),
    "recovery_windows": ("alert", [3, 6, 12, 24, 48]),
    "escalation_ratio": ("alert", [1.25, 1.5, 2.0, 3.0, 5.0]),
    "escalation_windows": ("alert", [1, 2, 3, 4]),
}

sweep_results = {}
for parameter, (owner, values) in SWEEPS.items():
    key = "model_update" if owner == "model" else "alert_update"
    sweep_results[parameter] = pd.DataFrame({v: metrics_for(**{key: {parameter: v}}) for v in values}).T

In [ ]:
fig = make_subplots(rows=2, cols=3, subplot_titles=[f"<b>{p}</b>" for p in SWEEPS],
                    vertical_spacing=0.18, horizontal_spacing=0.07)
for position, (parameter, (owner, values)) in enumerate(SWEEPS.items()):
    row, col = divmod(position, 3)
    table = sweep_results[parameter]
    for metric, color in METRIC_COLORS.items():
        fig.add_trace(go.Scatter(
            x=[str(v) for v in table.index], y=table[metric], mode="lines+markers", legendgroup=metric,
            name=METRIC_LABELS[metric], showlegend=position == 0,
            line=dict(color=color, width=2), marker=dict(size=8, line=dict(color="white", width=1)),
            hovertemplate="%{y:.2f}<extra>" + METRIC_LABELS[metric] + "</extra>",
        ), row=row + 1, col=col + 1)
    current = getattr(PCA_PARAMS if owner == "model" else WINDOW_ALERT_PARAMS, parameter)
    # Num eixo de categorias o x de uma forma é a *posição* da categoria; um rótulo numérico como "2.0"
    # seria lido como posição 2, então a posição do valor escolhido é passada explicitamente.
    chosen = [float(v) for v in values].index(float(current))
    fig.add_vline(x=chosen, line=dict(color=NEUTRAL, width=1, dash="dot"), row=row + 1, col=col + 1)
    fig.update_xaxes(type="category", row=row + 1, col=col + 1)
style(fig, "Precisão, recall e F1 conforme um hiperparâmetro varia", height=640,
      subtitle="Detector PCA nas janelas corrigidas; a linha pontilhada marca o valor em hyperparameters/*.yaml")
fig.update_layout(hovermode="x unified")
fig.update_annotations(font_size=13)
fig.update_yaxes(range=[0, 1])
fig.show()

display(pd.concat(sweep_results).round(3))

### Intervalo pontuado e passo

Com as janelas corrigidas, a janela do pipeline deixa de definir a escala de tempo; quem define é `window_hours`, tanto para o baseline aprendido no `fit` quanto para o trecho pontuado em cada janela. A última linha divide o passo pela metade (janelas de 12 h, sobreposição de 11,5 h) e dobra as contagens do motor de alertas para que continuem cobrindo o mesmo número de horas.

In [ ]:
half_step_windows = slice_windows(PipelineParams(model_window_size_hours=12, window_overlap_hours=11.5))
half_step_alerts = WINDOW_ALERT_PARAMS.model_copy(update={
    "onset_windows": 2 * WINDOW_ALERT_PARAMS.onset_windows,
    "escalation_windows": 2 * WINDOW_ALERT_PARAMS.escalation_windows,
    "recovery_windows": 2 * WINDOW_ALERT_PARAMS.recovery_windows,
})
span_sweep = pd.DataFrame({
    **{f"{hours} h pontuadas, passo de 1 h": global_metrics(raise_alerts(
        score_pca(PCA_PARAMS.model_copy(update={"window_hours": hours}))))
       for hours in (1, 2, 3, 4, 6)},
    f"{PCA_PARAMS.window_hours:g} h pontuadas, passo de 30 min": global_metrics(raise_alerts(
        score_pca(windows=half_step_windows), lambda: WindowCountAlertEngine(params=half_step_alerts))),
}).T
span_sweep[["TP", "FP", "FN"]] = span_sweep[["TP", "FP", "FN"]].astype(int)
display(span_sweep.round(3))

## 11. O que foi testado e não ajudou: features além do nível da janela

O EDA mostrou incidentes em que o nível quase não se mexe enquanto outras propriedades da janela mudam. Três resumos foram testados ao lado do nível da janela usado pelo PCA:

- **pico** — máximo de cada canal em log na janela (surtos curtos);
- **dispersão** — intervalo interquartil de cada canal em log;
- **razão vel/acc** — mediana de log(velocidade) − log(aceleração) por eixo, um indicador de mudança no conteúdo de frequência (bilateral).

Eles entram no **mesmo PCA** do nível (um modelo conjunto) ou num **PCA separado**, cuja severidade é combinada com a do nível pelo máximo, com margem própria. As últimas linhas dispensam o PCA e marcam a janela pelo maior z de um único canal, para medir quanto vale a correlação entre os eixos. Todas as linhas usam as janelas corrigidas e o motor de alertas padrão.

In [ ]:
QUANTILE = 1 - PCA_PARAMS.window_anomaly_ratio
SPAN = pd.Timedelta(hours=PCA_PARAMS.window_hours)
FLOORS = {"level": 0.1, "peak": 0.1, "spread": 0.05, "ratio": 0.1}
KIND_LABELS = {"level": "nível", "peak": "pico", "spread": "dispersão", "ratio": "razão vel/acc"}


def log_running(samples: TimeSeries) -> pd.DataFrame:
    running = [p for p in samples.data if p.uptime]
    values = np.log(np.maximum([[getattr(p, c) for c in FEATURE_NAMES] for p in running], 1e-6)).reshape(-1, 6)
    return pd.DataFrame(values, index=pd.DatetimeIndex([p.timestamp for p in running]), columns=list(FEATURE_NAMES))


def rolling_summary(X: pd.DataFrame, kind: str) -> pd.DataFrame:
    if kind == "ratio":
        X = pd.DataFrame(X[VELOCITY].to_numpy() - X[ACCELERATION].to_numpy(), index=X.index)
        return X.rolling(SPAN, min_periods=PCA_PARAMS.min_running_samples).median()
    rolling = X.rolling(SPAN, min_periods=PCA_PARAMS.min_running_samples)
    return {"level": lambda: rolling.quantile(QUANTILE), "peak": rolling.max,
            "spread": lambda: rolling.quantile(0.75) - rolling.quantile(0.25)}[kind]()


def window_summary(v: np.ndarray, kind: str) -> np.ndarray:
    if kind == "ratio":
        return np.median(v[:, :3] - v[:, 3:], axis=0)
    return {"level": lambda: np.quantile(v, QUANTILE, axis=0), "peak": lambda: v.max(axis=0),
            "spread": lambda: np.quantile(v, 0.75, axis=0) - np.quantile(v, 0.25, axis=0)}[kind]()


WINDOW_LOGS = {i: [log_running(scoring_span(w)) for w in WINDOWS["gap_proof"][i]] for i in SCENARIOS}
FIT_LOGS = {i: log_running(fit) for i, (fit, _, _) in SCENARIOS.items()}


def stream(kinds: tuple, detector: str = "pca") -> dict:
    """(score / limite do treino, aponta para cima) para cada janela corrigida, um modelo sobre `kinds`."""
    out = {}
    for i in SCENARIOS:
        L = pd.concat([rolling_summary(FIT_LOGS[i], k) for k in kinds], axis=1).dropna().to_numpy()
        center = np.median(L, axis=0)
        floors = np.concatenate([[FLOORS[k]] * (3 if k == "ratio" else 6) for k in kinds])
        scale = np.maximum(1.4826 * np.median(np.abs(L - center), axis=0), floors)
        upward = np.concatenate([[k != "ratio"] * (3 if k == "ratio" else 6) for k in kinds])
        Z = (L - center) / scale
        inliers = np.abs(Z).max(axis=1) <= PCA_PARAMS.fit_outlier_z
        Z = Z[inliers] if inliers.sum() >= len(Z) // 2 else Z
        if detector == "pca":
            pca = _PcaDetector(PCA_PARAMS)
            pca.fit(Z)
            score = lambda z: pca.score(z)
        else:  # maior z de um único canal, sem modelo de correlação
            score = lambda z: ((z**2).max(axis=1), z**2)
        limit = np.quantile(score(Z)[0], PCA_PARAMS.threshold_quantile)
        results = []
        for v in WINDOW_LOGS[i]:
            if len(v) < PCA_PARAMS.min_running_samples:
                results.append(None)
                continue
            z = (np.concatenate([window_summary(v.to_numpy(), k) for k in kinds]) - center) / scale
            s, contributions = score(z[np.newaxis, :])
            direction = np.where(upward, np.sign(z), 1.0)  # resumos bilaterais sempre contam como piora
            results.append((float(s[0]) / limit, float(contributions[0] @ direction) > 0))
        out[i] = results
    return out


def fused_metrics(streams: list) -> dict:
    """Severidade = máximo sobre (stream, margem); anormal se algum stream passa da margem e aponta para cima."""
    outputs = {}
    for i in SCENARIOS:
        outputs[i] = []
        for k, window in enumerate(WINDOWS["gap_proof"][i]):
            timestamp = window.data[-1].timestamp
            values = [(np.sqrt(max(s[i][k][0], 0) / margin), s[i][k][1]) for s, margin in streams if s[i][k] is not None]
            if not values:
                outputs[i].append(PredictOutput(anomaly_status=False, timestamp=timestamp))
                continue
            upward = [sev for sev, up in values if up]
            severity = max(upward) if upward else max(sev for sev, _ in values)
            outputs[i].append(PredictOutput(anomaly_status=bool(upward and max(upward) >= 1),
                                            timestamp=timestamp, severity=max(severity, 1e-6)))
    return global_metrics(raise_alerts(outputs))


margin = PCA_PARAMS.threshold_margin
level = stream(("level",))
feature_tests = {"Só nível (o PCA acima)": fused_metrics([(level, margin)])}
for kinds in (("level", "peak"), ("level", "spread"), ("level", "ratio"), ("level", "peak", "spread", "ratio")):
    feature_tests[f"Mesmo PCA: {' + '.join(KIND_LABELS[k] for k in kinds)}"] = fused_metrics([(stream(kinds), margin)])
for kind in ("peak", "spread", "ratio"):
    extra = stream((kind,))
    for extra_margin in (3, 6, 9):
        feature_tests[f"Nível + modelo separado de {KIND_LABELS[kind]} (margem {extra_margin})"] = fused_metrics(
            [(level, margin), (extra, extra_margin)])
single = stream(("level",), detector="max_z")
for single_margin in (2, 3):
    feature_tests[f"Nível, maior z de um canal sem PCA (margem {single_margin})"] = fused_metrics([(single, single_margin)])

feature_tests = pd.DataFrame(feature_tests).T
feature_tests[["TP", "FP", "FN"]] = feature_tests[["TP", "FP", "FN"]].astype(int)
display(feature_tests.round(3))

**Nenhum dos resumos extras ajuda.**

- **No mesmo PCA**, cada um deles reduz o F1 (0,35–0,45 contra 0,56). Eles também variam muito durante a operação normal no período de predição, então o modelo conjunto passa a ver mais janelas comuns como anormais. Os eventos abrem cedo, antes da janela rotulada, onde o alerta conta como falso positivo e o incidente fica dependendo de uma escalada que pode não vir.
- **Como modelos separados**, eles se aproximam por baixo do resultado só com o nível à medida que a margem sobe (melhor: 0,52) e nunca acrescentam um incidente líquido dos que custam.
- **Sem o PCA**, marcar a janela pelo maior z de um único canal é claramente pior (0,26–0,40): é a correlação entre os eixos que permite ao PCA distinguir uma mudança da máquina de um canal ruidoso.

A tabela de intervalo pontuado da seção 10 mostra o mesmo para a escala de tempo: 2 h pontuadas a cada hora é o melhor dos intervalos testados, e dividir o passo pela metade não ajuda (0,55).

## Seções 12 a 15 — testes adicionais sobre o baseline ajustado

As quatro seções seguintes partem do **baseline ajustado nas janelas corrigidas** (TP 16, FP 12, FN 13, F1 0,56) e perguntam o que ainda pode melhorá-lo sem sair da sua metodologia: um z por leitura contra o baseline do ativo e uma regra sobre a janela.

- **12** — outras features da janela (RMS, pico, fator de crista, curtose…) ajudam?
- **13** — o limite de 4,5 desvios-padrão é a escolha adequada?
- **14** — um limiar dinâmico (média móvel + desvio-padrão móvel) é melhor que o limiar fixo do treino?
- **15** — onde estão os erros que sobram, e que outras abordagens parecem promissoras?

Todas as variantes usam as mesmas janelas corrigidas, o mesmo motor de alertas e a mesma pontuação. Para testar dezenas de variantes sem refazer os objetos do pipeline, a célula abaixo guarda as leituras com a máquina rodando de cada cenário em arrays e reescreve a regra do baseline sobre eles; um `assert` confere que o resultado é idêntico ao do `AdjustedBaseline`, janela por janela.

**Quanto vale uma diferença de F1.** São 29 cenários e 29 incidentes, então um alerta a mais ou a menos move o F1 em cerca de 0,02. Reamostrando os cenários (bootstrap), o F1 do baseline ajustado fica entre 0,44 e 0,68 em 90% das vezes. Diferenças abaixo de uns 0,05 nas tabelas a seguir devem ser lidas como empate; a seção 15 mede isso para as melhores candidatas.

Nas tabelas, **ganhos** e **perdidos** listam os incidentes que a variante encontra a mais ou a menos que o baseline ajustado (número do cenário, com `a`/`b` quando o cenário tem dois incidentes).

In [ ]:
from functools import lru_cache
from typing import Callable, NamedTuple

K_ADJUSTED = MODELS["adjusted"][1].z_threshold  # limite do baseline ajustado (4,5)
RATIO = PCA_PARAMS.window_anomaly_ratio
MIN_SAMPLES = PCA_PARAMS.min_running_samples
SPAN_HOURS = PCA_PARAMS.window_hours
MAD_TO_SIGMA = 1.4826


class Span(NamedTuple):
    """O que uma janela corrigida entrega ao modelo, como posições nos arrays do cenário."""

    timestamp: datetime  # instante da decisão (última leitura da janela)
    window_start: int    # primeira leitura rodando da janela de 12 h
    start: int           # primeira leitura rodando das últimas 2 h, o trecho pontuado
    stop: int            # fim (exclusivo) dos dois trechos


class ScenarioArrays(NamedTuple):
    """Leituras com a máquina rodando de um cenário, treino seguido de predição."""

    scenario_id: int
    times: np.ndarray     # instante de cada leitura
    channels: np.ndarray  # [n, 6] amplitude dos seis canais
    norms: np.ndarray     # [n, 2] norma da velocidade e norma da aceleração
    n_fit: int            # as primeiras n_fit leituras são do treino
    spans: list           # um Span por janela corrigida


Rule = Callable[[Span], tuple]  # janela -> (anormal, severidade)


def hours(value: float) -> np.timedelta64:
    return np.timedelta64(int(value * 3600), "s")


def to_naive(timestamp: datetime) -> np.datetime64:
    return pd.Timestamp(timestamp).tz_convert(None).to_datetime64()


def to_arrays(scenario_id: int) -> ScenarioArrays:
    fit, prediction, _ = SCENARIOS[scenario_id]
    fit_points = [p for p in fit.data if p.uptime]
    points = fit_points + [p for p in prediction.data if p.uptime]
    times = pd.DatetimeIndex([p.timestamp for p in points]).tz_convert(None).to_numpy()
    channels = np.array([[getattr(p, c) for c in FEATURE_NAMES] for p in points], dtype=float)
    norms = np.column_stack([np.linalg.norm(channels[:, :3], axis=1), np.linalg.norm(channels[:, 3:], axis=1)])

    spans = []
    for window in WINDOWS["gap_proof"][scenario_id]:
        first, last = to_naive(window.data[0].timestamp), to_naive(window.data[-1].timestamp)
        window_start = max(int(np.searchsorted(times, first, "left")), len(fit_points))
        start = max(int(np.searchsorted(times, last - hours(SPAN_HOURS), "right")), window_start)
        spans.append(Span(window.data[-1].timestamp, window_start, start, int(np.searchsorted(times, last, "right"))))
    return ScenarioArrays(scenario_id, times, channels, norms, len(fit_points), spans)


ARRAYS = {scenario_id: to_arrays(scenario_id) for scenario_id in SCENARIOS}


def share_rule(z: np.ndarray, k, ratio: float = RATIO) -> tuple:
    """A regra do baseline: anormal quando `ratio` das leituras passam de `k` em alguma coluna.

    A severidade é o percentil (1 - ratio) do z na pior coluna, em unidades do limite.
    """
    abnormal = bool(((z > k).mean(axis=0) >= ratio).any())
    return abnormal, float((np.quantile(z, 1 - ratio, axis=0) / k).max())


def fit_mean_std(arrays: ScenarioArrays) -> tuple:
    fit = arrays.norms[: arrays.n_fit]
    return fit.mean(axis=0), np.maximum(fit.std(axis=0), EPS)


def norm_z(arrays: ScenarioArrays) -> np.ndarray:
    """z das duas normas em cada leitura, contra a média e o desvio-padrão do treino."""
    center, scale = fit_mean_std(arrays)
    return (arrays.norms - center) / scale


def reading_rule(k=K_ADJUSTED, ratio: float = RATIO, represent=lambda arrays: arrays.norms, robust: bool = False):
    """Regra do baseline ajustado: z de cada leitura contra o treino, janela julgada por `share_rule`.

    `represent` escolhe as colunas julgadas (as duas normas por padrão) e `robust` troca média e
    desvio-padrão por mediana e MAD. `k` é um número ou uma função (z, arrays) -> limite por coluna.
    """
    def make(arrays: ScenarioArrays) -> Rule:
        values = represent(arrays)
        fit = values[: arrays.n_fit]
        if robust:
            center = np.median(fit, axis=0)
            scale = MAD_TO_SIGMA * np.median(np.abs(fit - center), axis=0)
        else:
            center, scale = fit.mean(axis=0), fit.std(axis=0)
        z = (values - center) / np.maximum(scale, EPS)
        limit = k(z, arrays) if callable(k) else k
        return lambda span: share_rule(z[span.start : span.stop], limit, ratio)
    return make


def score_windows(make_rule) -> dict:
    """Saída de cada janela corrigida; `make_rule(arrays)` devolve a regra de um cenário."""
    outputs = {}
    for scenario_id, arrays in ARRAYS.items():
        rule = make_rule(arrays)
        outputs[scenario_id] = []
        for span in arrays.spans:
            n_valid = span.stop - span.start
            # Sem severidade: máquina (quase toda) parada, sem evidência; o motor de alertas mantém o estado.
            output = PredictOutput(anomaly_status=False, timestamp=span.timestamp, n_valid=n_valid)
            if n_valid >= MIN_SAMPLES:
                abnormal, severity = rule(span)
                output = output.model_copy(
                    update={"anomaly_status": bool(abnormal), "severity": max(float(severity), 1e-6)})
            outputs[scenario_id].append(output)
    return outputs


def found_incidents(decisions: dict) -> set:
    """Incidentes com algum alerta dentro, rotulados pelo cenário (mais a/b quando o cenário tem dois)."""
    found = set()
    for scenario_id, scenario_decisions in decisions.items():
        alerts = [d.timestamp for d in scenario_decisions if d.alert]
        incidents = SCENARIOS[scenario_id][2]
        for position, incident in enumerate(incidents):
            if any(incident.start <= alert <= incident.end for alert in alerts):
                found.add(f"{scenario_id}{'ab'[position] if len(incidents) > 1 else ''}")
    return found


def incident_order(label: str) -> tuple:
    return int(label.rstrip("ab")), label


def join_incidents(labels: set) -> str:
    return ", ".join(sorted(labels, key=incident_order))


def compare(variants: dict) -> pd.DataFrame:
    """Uma linha por variante (nome -> decisões): placar e incidentes ganhos e perdidos contra o baseline ajustado."""
    rows = {}
    for variant, decisions in variants.items():
        found = found_incidents(decisions)
        rows[variant] = {**global_metrics(decisions), "ganhos": join_incidents(found - BASE_FOUND),
                         "perdidos": join_incidents(BASE_FOUND - found)}
    return pd.DataFrame.from_dict(rows, orient="index").round(3)


def grid_cell(metrics: dict) -> str:
    """Placar numa célula de tabela: F1 · TP/FP."""
    return f"{metrics['f1']:.2f} · {metrics['TP']}/{metrics['FP']}"


def scenario_counts(decisions: dict) -> np.ndarray:
    """[n_cenários, 3] com TP, FP e FN de cada cenário."""
    return np.array([[evaluate(SCENARIOS[i][2], decisions[i])[c] for c in ("TP", "FP", "FN")] for i in SCENARIOS])


def f1_score(counts: np.ndarray) -> np.ndarray:
    """F1 global a partir de contagens TP, FP, FN na última dimensão."""
    tp, fp, fn = np.moveaxis(counts, -1, 0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


# Reamostragens dos cenários, as mesmas para toda variante, o que permite comparar duas variantes par a par.
RESAMPLES = np.random.default_rng(0).integers(len(SCENARIOS), size=(4000, len(SCENARIOS)))


def bootstrap_f1(decisions: dict) -> np.ndarray:
    """F1 global em cada reamostragem dos cenários."""
    return f1_score(scenario_counts(decisions)[RESAMPLES].sum(axis=1))


BASE_OUTPUTS = score_windows(reading_rule())
BASE_DECISIONS = raise_alerts(BASE_OUTPUTS)
BASE_FOUND = found_incidents(BASE_DECISIONS)
BASE_F1 = bootstrap_f1(BASE_DECISIONS)

same_status = all(
    ours.anomaly_status == reference.anomaly_status
    for i in SCENARIOS for ours, reference in zip(BASE_OUTPUTS[i], main_outputs("adjusted")[i])
)
assert same_status and global_metrics(BASE_DECISIONS) == global_metrics(main_decisions("adjusted")), (
    "a regra em arrays deveria reproduzir o baseline ajustado")

print("Baseline ajustado refeito em arrays:", global_metrics(BASE_DECISIONS))
low, high = np.quantile(BASE_F1, [0.05, 0.95])
print(f"F1 por bootstrap sobre os cenários: intervalo de 90% de {low:.2f} a {high:.2f}")

## 12. Features da janela na regra do baseline

O baseline ajustado olha uma coisa só: quantas leituras da janela passam do limite. Features clássicas de monitoramento de vibração (RMS, pico, fator de crista, curtose) descrevem outras propriedades do sinal. Duas ressalvas antes de testá-las:

- **Os dados já são RMS.** Cada leitura é o RMS de um trecho de sinal, uma a cada ~10 min; a forma de onda não está disponível. Fator de crista e curtose *do sinal*, que denunciam impactos de rolamento, não podem ser calculados. O que dá para calcular são estatísticas da **tendência do RMS dentro da janela de 2 h** (cerca de 12 leituras): elas dizem se o nível oscilou ou teve surtos na janela, não qual é o formato da vibração.
- **Mesma metodologia do baseline.** Cada estatística é calculada para a norma da velocidade e para a norma da aceleração e comparada com a sua média e o seu desvio-padrão nas janelas de 2 h do treino. A janela é anormal quando o z da estatística passa de um limite em alguma das normas, só para cima.

| Estatística | O que mede na janela |
|---|---|
| média, RMS | nível |
| pico | maior leitura |
| fator de crista | pico ÷ RMS: um surto isolado sobre um nível estável |
| desvio-padrão, coef. de variação, pico a pico | oscilação do nível, absoluta ou relativa |
| curtose, assimetria | forma da distribuição das leituras |

Cada estatística é testada **sozinha** e **somada à regra do baseline** (a janela é anormal se a regra do baseline *ou* a estatística disparar).

In [ ]:
def standardized_moment(x: np.ndarray, order: int) -> np.ndarray:
    centered = x - x.mean(axis=0)
    return (centered**order).mean(axis=0) / np.maximum(centered.std(axis=0), EPS) ** order


def window_rms(x: np.ndarray) -> np.ndarray:
    return np.sqrt((x**2).mean(axis=0))


# Estatísticas da tendência do RMS dentro da janela, uma por norma.
WINDOW_STATS = {
    "média": lambda x: x.mean(axis=0),
    "RMS": window_rms,
    "pico": lambda x: x.max(axis=0),
    "fator de crista": lambda x: x.max(axis=0) / window_rms(x),
    "desvio-padrão": lambda x: x.std(axis=0),
    "coef. de variação": lambda x: x.std(axis=0) / x.mean(axis=0),
    "pico a pico": lambda x: x.max(axis=0) - x.min(axis=0),
    "curtose": lambda x: standardized_moment(x, 4) - 3,
    "assimetria": lambda x: standardized_moment(x, 3),
}


def fit_spans(arrays: ScenarioArrays) -> list:
    """(início, fim) de uma janela de 2 h terminando em cada leitura do treino, quando tem leituras suficientes."""
    times = arrays.times[: arrays.n_fit]
    starts = np.searchsorted(times, times - hours(SPAN_HOURS), "right")
    return [(start, stop) for stop, start in enumerate(starts, start=1) if stop - start >= MIN_SAMPLES]


@lru_cache(maxsize=None)
def stat_baseline(stat: str, scenario_id: int) -> tuple:
    """Média e desvio-padrão da estatística nas janelas de 2 h do treino."""
    norms = ARRAYS[scenario_id].norms
    values = np.array([WINDOW_STATS[stat](norms[start:stop]) for start, stop in fit_spans(ARRAYS[scenario_id])])
    return values.mean(axis=0), np.maximum(values.std(axis=0), EPS)


@lru_cache(maxsize=None)
def stat_z(stat: str, scenario_id: int, start: int, stop: int) -> float:
    """z da estatística da janela, na pior das duas normas."""
    center, scale = stat_baseline(stat, scenario_id)
    return float(((WINDOW_STATS[stat](ARRAYS[scenario_id].norms[start:stop]) - center) / scale).max())


def stat_rule(stat: str, limit: float, with_baseline: bool):
    """Janela anormal quando o z da estatística passa de `limit`, sozinha ou somada (OU) à regra do baseline."""
    def make(arrays: ScenarioArrays) -> Rule:
        baseline = reading_rule()(arrays)

        def rule(span: Span) -> tuple:
            z = stat_z(stat, arrays.scenario_id, span.start, span.stop)
            abnormal, severity = z > limit, z / limit
            if with_baseline:
                baseline_abnormal, baseline_severity = baseline(span)
                abnormal, severity = abnormal or baseline_abnormal, max(severity, baseline_severity)
            return abnormal, severity
        return rule
    return make


STAT_LIMITS = {"sozinha": (3, 4.5, 6, 8), "com a regra do baseline": (4.5, 6, 8, 12)}
STAT_DECISIONS = {
    (mode, stat, limit): raise_alerts(score_windows(stat_rule(stat, limit, with_baseline=mode != "sozinha")))
    for mode, limits in STAT_LIMITS.items() for stat in WINDOW_STATS for limit in limits
}

for mode, limits in STAT_LIMITS.items():
    print(f"Estatística {mode} — F1 · TP/FP para cada limite (em desvios-padrão das janelas do treino):")
    display(pd.DataFrame({
        f"limite {limit:g}": {
            stat: grid_cell(global_metrics(STAT_DECISIONS[(mode, stat, limit)])) for stat in WINDOW_STATS}
        for limit in limits
    }))

print("Estatística com a regra do baseline, limite 6 — o que cada uma ganha e perde:")
display(compare({stat: STAT_DECISIONS[("com a regra do baseline", stat, 6)] for stat in WINDOW_STATS}))

**Nenhuma estatística melhora o baseline.**

- **As de nível repetem o que a regra já vê.** Média, RMS e pico sozinhos chegam no máximo a F1 0,48–0,52, abaixo do 0,56 da regra do baseline, que usa a mesma informação de forma mais robusta (a fração de leituras acima do limite ignora surtos isolados).
- **As de forma não têm sinal.** Curtose e assimetria não encontram nenhum incidente em nenhum limite; fator de crista e coeficiente de variação ficam em F1 0,06–0,12. Com ~12 leituras por janela essas estatísticas são quase só ruído de amostragem, e uma falha que eleva todas as leituras da janela não muda a *forma* da distribuição.
- **As de oscilação pioram quando somadas.** Desvio-padrão, pico a pico e fator de crista abrem eventos em janelas agitadas mas normais. Com limites baixos custam de 5 a 13 falsos positivos a mais e ainda perdem incidentes (5b, 6b, 28a), porque o evento aberto cedo demais já está em curso quando o incidente começa. Subindo o limite elas param de disparar e o resultado volta ao do baseline.
- **O único ganho é a média com limite 6:** um incidente a mais (7b) por um falso positivo a mais, F1 0,58. É a diferença de um alerta, dentro do ruído.

A tabela seguinte muda as **colunas julgadas** pela regra, sem acrescentar estatísticas: os seis canais no lugar das duas normas, a amplitude em log, e mediana e MAD no lugar de média e desvio-padrão.

In [ ]:
REPRESENTATIONS = {
    "duas normas": lambda arrays: arrays.norms,
    "seis canais": lambda arrays: arrays.channels,
    "log das duas normas": lambda arrays: np.log(np.maximum(arrays.norms, 1e-6)),
    "log dos seis canais": lambda arrays: np.log(np.maximum(arrays.channels, 1e-6)),
}
SCALES = {"média e desvio-padrão": False, "mediana e MAD": True}

representation_tests = pd.DataFrame({
    f"k = {k:g}": {
        (representation, scale): grid_cell(global_metrics(raise_alerts(score_windows(
            reading_rule(k, represent=represent, robust=robust)))))
        for representation, represent in REPRESENTATIONS.items() for scale, robust in SCALES.items()
    }
    for k in (3, 4.5, 6, 8, 10, 14)
}).rename_axis(["colunas julgadas", "baseline"])
display(representation_tests)

- **Seis canais no lugar das duas normas:** pior em quase todos os limites (0,36–0,48). Seis colunas são seis chances de um canal ruidoso disparar a regra. Só um limite de 10 desvios compensa isso (F1 0,58, com 5 falsos positivos), mas os vizinhos (0,48 em k = 8 e 0,40 em k = 14) mostram um pico isolado, não um platô.
- **Logaritmo:** pior (no máximo 0,53). O log comprime justamente a cauda superior, onde estão as falhas; a regra por leitura passa a precisar de limites baixos e perde incidentes.
- **Mediana e MAD:** pior (0,30–0,48 nas duas normas). O MAD ignora as caudas do treino, o sigma fica menor e o mesmo k passa a disparar com a variação comum (28 falsos positivos em k = 4,5).

A conclusão é a mesma da seção 11 para o PCA: com leituras de RMS a cada 10 min, a informação útil está no nível, e as duas normas com média e desvio-padrão são a forma mais estável de lê-lo.

## 13. O limite de 4,5 desvios-padrão é adequado?

Quatro perguntas, da mais simples à mais exigente:

1. **Há um limite melhor?** Grade do limite k e da fração da janela.
2. **O mesmo k significa a mesma coisa em todo ativo?** Comportamento da regra no período de treino de cada ativo, e limites calculados por ativo.
3. **Quanto do F1 vem de ter escolhido k olhando os rótulos?** Validação deixando um cenário de fora.
4. **Até onde um limite perfeito levaria?** Um k por cenário, escolhido com os rótulos (oráculo).

In [ ]:
K_GRID = (3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 10)
RATIO_GRID = (0.1, 0.2, 0.3, 0.4, 0.5)
GRID_DECISIONS = {
    (k, ratio): raise_alerts(score_windows(reading_rule(k, ratio))) for k in K_GRID for ratio in RATIO_GRID}
grid = pd.DataFrame({ratio: {k: global_metrics(GRID_DECISIONS[(k, ratio)]) for k in K_GRID} for ratio in RATIO_GRID})

fig = go.Figure(go.Heatmap(
    x=[f"{ratio:.0%}" for ratio in RATIO_GRID], y=[f"{k:g}" for k in K_GRID],
    z=grid.map(lambda m: m["f1"]).values, zmin=0.2, zmax=0.6, colorscale=SEQUENTIAL,
    text=grid.map(lambda m: f"{m['f1']:.2f}<br>{m['TP']}/{m['FP']}").values, texttemplate="%{text}",
    xgap=2, ygap=2, colorbar=dict(title="F1"),
    hovertemplate="k = %{y}, fração = %{x}: F1 %{z:.2f}<extra></extra>",
))
style(fig, "F1 do baseline ajustado conforme o limite e a fração da janela", height=620, legend=False,
      subtitle="Em cada célula: F1 e TP/FP. O baseline ajustado usa k = 4,5 e 20%")
fig.update_layout(hovermode="closest")
fig.update_xaxes(title_text="Fração das leituras que precisa passar do limite", type="category")
fig.update_yaxes(title_text="Limite k (desvios-padrão do treino)", type="category", autorange="reversed")
fig.show()

**k = 4,5 com 20% é o melhor ponto da grade, num platô curto.** k = 4,5 e k = 5 dão o mesmo placar (16/12, F1 0,56). A partir de 5,5 o recall cai, e abaixo de 4 os falsos positivos sobem depressa (29 em k = 3). Nenhuma outra fração melhora: 10% deixa surtos curtos dispararem a regra, e 30% ou mais exige quase a janela inteira elevada e perde incidentes curtos. O resto da grade fica entre 0,28 e 0,55.

### O mesmo k em ativos diferentes

A tabela mostra como a regra se comporta no período de treino de cada ativo: os percentis do z das leituras, a curtose (0 numa distribuição normal) e a fração das janelas de 2 h do próprio treino que a regra com k = 4,5 marcaria como anormais.

In [ ]:
def calibration_row(arrays: ScenarioArrays) -> dict:
    """Como a regra com k = 4,5 se comporta no próprio período de treino de um ativo."""
    z = norm_z(arrays)
    fit_z = z[: arrays.n_fit]
    flagged = [share_rule(z[start:stop], K_ADJUSTED)[0] for start, stop in fit_spans(arrays)]
    return {
        "leituras_treino": arrays.n_fit,
        "z_p99": np.quantile(fit_z, 0.99, axis=0).max(),
        "z_p99.9": np.quantile(fit_z, 0.999, axis=0).max(),
        "z_máx": fit_z.max(),
        "curtose": (standardized_moment(fit_z, 4) - 3).max(),
        "janelas_do_treino_anormais_%": 100 * np.mean(flagged),
    }


calibration = pd.DataFrame({i: calibration_row(arrays) for i, arrays in ARRAYS.items()}).T.rename_axis("cenário")
calibration["leituras_treino"] = calibration["leituras_treino"].astype(int)
display(calibration.round(2))
print("Resumo entre os ativos:")
display(calibration.drop(columns="leituras_treino").describe().loc[["min", "25%", "50%", "75%", "max"]].round(2))

Se as caudas diferem tanto, o natural é calcular o limite por ativo: um quantil alto do z do treino, vezes uma margem de segurança. Duas versões, cada uma com um limite por norma: o quantil das **leituras** do treino e o quantil do **nível das janelas** de 2 h do treino (o percentil 80 do z, que é o que a regra de 20% compara com o limite).

In [ ]:
def fit_levels(z: np.ndarray, arrays: ScenarioArrays) -> np.ndarray:
    """Nível (percentil 80 do z) de cada janela de 2 h do treino."""
    return np.array([np.quantile(z[start:stop], 1 - RATIO, axis=0) for start, stop in fit_spans(arrays)])


def empirical_limit(quantile: float, margin: float, of_levels: bool):
    """Limite por ativo e por norma: `margin` × um quantil do z do treino (das leituras ou do nível das janelas)."""
    def limit(z: np.ndarray, arrays: ScenarioArrays) -> np.ndarray:
        reference = fit_levels(z, arrays) if of_levels else z[: arrays.n_fit]
        return np.maximum(margin * np.quantile(reference, quantile, axis=0), 1.0)
    return limit


EMPIRICAL = {"quantil das leituras do treino": False, "quantil do nível das janelas do treino": True}
empirical_tests = pd.DataFrame({
    f"margem {margin:g}": {
        (source, f"{quantile:.1%}" if quantile < 1 else "máximo"): grid_cell(global_metrics(raise_alerts(score_windows(
            reading_rule(k=empirical_limit(quantile, margin, of_levels))))))
        for source, of_levels in EMPIRICAL.items() for quantile in (0.99, 0.999, 1.0)
    }
    for margin in (1, 1.25, 1.5, 2, 3)
}).rename_axis(["limite por ativo", "quantil"])
display(empirical_tests)

**O mesmo k não significa a mesma coisa em todo ativo, mas calcular o limite por ativo não resolve.**

- A maior leitura do treino fica a 1,3 desvios da média no cenário 20 e a 43,8 no cenário 17; o percentil 99 vai de 1,2 a 6,3. Em ativos de cauda leve o limite de 4,5 nunca é tocado no treino; nos de cauda pesada (cenários 2, 5, 7 e 8) a regra marcaria de 1% a 3% das janelas do próprio treino.
- Mesmo assim, as 30 variantes de limite por ativo ficam todas abaixo do limite único: de 0,38 a 0,54, contra 0,56. Um quantil alto de um treino de 4 a 31 dias é uma estimativa instável (o máximo depende de uma única leitura), e o período de predição não se parece com o de treino o bastante para que a cauda de um descreva a do outro.

### Validação e teto

A primeira tabela repete a escolha do limite deixando um cenário de fora por vez: a configuração é escolhida nos outros 28 e aplicada ao que ficou de fora. A segunda dá o melhor k de cada cenário quando ele é escolhido olhando os rótulos do próprio cenário, que é um teto e não um método.

In [ ]:
GRID_COUNTS = {config: scenario_counts(decisions) for config, decisions in GRID_DECISIONS.items()}


def leave_one_out(candidates: list) -> dict:
    """Para cada cenário, escolhe a configuração de melhor F1 nos outros 28 e a aplica ao que ficou de fora."""
    held_out, chosen = [], []
    for position in range(len(SCENARIOS)):
        others = np.arange(len(SCENARIOS)) != position
        best = max(candidates, key=lambda config: f1_score(GRID_COUNTS[config][others].sum(axis=0)))
        held_out.append(GRID_COUNTS[best][position])
        chosen.append(best)
    tp, fp, fn = np.sum(held_out, axis=0)
    usual = max(set(chosen), key=chosen.count)
    return {"TP": tp, "FP": fp, "FN": fn, "f1": f1_score(np.sum(held_out, axis=0)),
            "configuração_mais_escolhida": f"k = {usual[0]:g}, {usual[1]:.0%}",
            "dobras_em_que_foi_escolhida": chosen.count(usual)}


in_sample = global_metrics(BASE_DECISIONS)
validation = pd.DataFrame.from_dict({
    "Na amostra (k = 4,5 e 20%, escolhidos nos 29 cenários)": {
        **{c: in_sample[c] for c in ("TP", "FP", "FN", "f1")},
        "configuração_mais_escolhida": "—", "dobras_em_que_foi_escolhida": "—"},
    "Fora da amostra, escolhendo só k": leave_one_out([(k, RATIO) for k in K_GRID]),
    "Fora da amostra, escolhendo k e a fração": leave_one_out(list(GRID_COUNTS)),
}, orient="index")
display(validation.round(3))

# Oráculo: o melhor k de cada cenário, escolhido olhando os rótulos do próprio cenário.
ORACLE_GRID = (2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 10, 12, 15, 20)
oracle_counts = np.stack([scenario_counts(raise_alerts(score_windows(reading_rule(k)))) for k in ORACLE_GRID])
best_k = np.argmax(oracle_counts[:, :, 0] - oracle_counts[:, :, 1] - 1e-3 * oracle_counts[:, :, 1], axis=0)
oracle = pd.DataFrame(oracle_counts[best_k, np.arange(len(SCENARIOS))],
                      index=pd.Index(SCENARIOS, name="cenário"), columns=["TP", "FP", "FN"])
oracle.insert(0, "melhor_k", [ORACLE_GRID[position] for position in best_k])

print(f"Oráculo com um k por cenário: TP {oracle['TP'].sum()}, FP {oracle['FP'].sum()}, FN {oracle['FN'].sum()}, "
      f"F1 {f1_score(oracle[['TP', 'FP', 'FN']].to_numpy().sum(axis=0)):.2f}")
display(oracle)

**O limite é estável, mas o F1 medido é otimista.** Deixando um cenário de fora, k = 4,5 é a escolha em 26 das 29 dobras, então ele não depende de um cenário em particular. Só que o F1 nos cenários deixados de fora é 0,47 (13/13/16), contra 0,56 na amostra, e cai para 0,41 quando a fração também é escolhida. Cerca de 0,09 do F1 do placar vem de o limite ter sido ajustado nos mesmos dados em que é medido.

**Um limite por cenário chegaria a F1 0,80, mas não a partir do treino.** O oráculo encontra 20 incidentes com 1 falso positivo, com limites que vão de 2 a 20 conforme o cenário. Existe bastante espaço acima do limite único, e ele está em adaptar o limite a cada ativo. O que falta é de onde tirar essa adaptação, já que as estatísticas do treino não a fornecem (tabela anterior). Em vários cenários o nível normal do período de predição já não é o do treino (`eda.ipynb`, seção 6), o que leva ao limiar dinâmico da seção 14.

**Resumo da seção.** Para um limite fixo e único, 4,5 desvios-padrão com 20% da janela é a escolha adequada: é o melhor ponto da grade, fica num platô e é escolhido de forma consistente fora da amostra. O que limita o resultado não é o valor do limite, e sim ele ser o mesmo para todo ativo e para todo o período de predição.

## 14. Limiar dinâmico: média móvel + desvio-padrão móvel

O baseline ajustado compara cada leitura com a média e o desvio-padrão **do período de treino**, que ficam fixos. A alternativa clássica é um limiar que acompanha os dados: a leitura é comparada com a média e o desvio-padrão das últimas *N* horas (`média móvel + k × desvio móvel`). O resto da regra não muda: a janela é anormal quando 20% das suas leituras passam do limiar.

Como o teste foi montado:

- A referência móvel usa as leituras com a máquina rodando das *N* horas anteriores ao trecho pontuado, sem incluir as 2 h que estão sendo julgadas. O histórico começa no fim do treino, então a primeira janela da predição já tem referência; com menos de 30 leituras, vale o baseline do treino.
- **Média e desvio móveis** é a forma direta. **Mediana e MAD móveis** é a versão menos sensível à própria falha. **Desvio móvel com piso** deixa o limiar acompanhar o nível, mas a faixa de tolerância nunca fica mais estreita que a do treino. **Fixo E móvel** exige que as duas regras disparem; **fixo OU móvel** aceita qualquer uma.
- No pipeline real isso exigiria que o modelo guardasse o histórico entre chamadas de `predict`, porque cada janela só entrega 12 h de dados.

In [ ]:
def recent_mean_std(arrays: ScenarioArrays, start: int, window_hours: float, robust: bool, min_readings: int = 30):
    """Centro e dispersão das leituras rodando nas `window_hours` antes do trecho pontuado; None se forem poucas.

    O histórico começa no fim do treino, então a primeira janela da predição já tem referência.
    """
    low = np.searchsorted(arrays.times, arrays.times[start - 1] - hours(window_hours), "left")
    recent = arrays.norms[low:start]
    if len(recent) < min_readings:
        return None
    if robust:
        center = np.median(recent, axis=0)
        return center, np.maximum(MAD_TO_SIGMA * np.median(np.abs(recent - center), axis=0), EPS)
    return recent.mean(axis=0), np.maximum(recent.std(axis=0), EPS)


def moving_rule(window_hours: float, k: float, robust: bool = False, floor: float = 0.0, combine: str | None = None):
    """Média móvel + k × desvio móvel no lugar da média e do desvio fixos do treino.

    `floor` é o piso do desvio móvel, em fração do desvio do treino. `combine` junta a regra móvel
    à regra fixa do baseline ajustado: "and" exige as duas, "or" aceita qualquer uma.
    """
    def make(arrays: ScenarioArrays) -> Rule:
        fit_center, fit_scale = fit_mean_std(arrays)
        fixed = reading_rule()(arrays)

        def rule(span: Span) -> tuple:
            center, scale = recent_mean_std(arrays, span.start, window_hours, robust) or (fit_center, fit_scale)
            scale = np.maximum(scale, floor * fit_scale)
            abnormal, severity = share_rule((arrays.norms[span.start : span.stop] - center) / scale, k)
            if combine is None:
                return abnormal, severity
            fixed_abnormal, fixed_severity = fixed(span)
            if combine == "and":
                return abnormal and fixed_abnormal, min(severity, fixed_severity)
            return abnormal or fixed_abnormal, max(severity, fixed_severity)
        return rule
    return make


def window_label(window_hours: float) -> str:
    return f"{window_hours:g} h" if window_hours < 168 else f"{window_hours / 24:g} d"


MOVING_FAMILIES = {
    "Média e desvio móveis": [dict(window_hours=w, k=k) for w in (24, 72, 168, 336) for k in (3, 4.5, 6)],
    "Mediana e MAD móveis": [dict(window_hours=w, k=k, robust=True) for w in (24, 72, 168, 336) for k in (4.5, 6, 8)],
    "Desvio móvel com piso no desvio do treino": [
        dict(window_hours=w, k=k, floor=1.0) for w in (72, 168, 336) for k in (3, 4.5)],
    "Fixo E móvel": [dict(window_hours=w, k=k, combine="and") for w in (72, 168, 336) for k in (2, 3)],
    "Fixo OU móvel (com piso)": [
        dict(window_hours=w, k=k, floor=1.0, combine="or") for w in (72, 168, 336) for k in (3, 4.5)],
}
MOVING_OUTPUTS = {
    (family, f"{window_label(config['window_hours'])}, k = {config['k']:g}"): score_windows(moving_rule(**config))
    for family, configs in MOVING_FAMILIES.items() for config in configs
}
moving_tests = compare({variant: raise_alerts(outputs) for variant, outputs in MOVING_OUTPUTS.items()})
moving_tests.index = pd.MultiIndex.from_tuples(moving_tests.index, names=["família", "janela e limite"])
display(moving_tests)

**Janelas móveis curtas pioram muito; só a janela longa com piso empata ou passa um pouco do limiar fixo.**

- **A janela curta absorve a própria falha.** Com 24 h de referência o F1 cai para 0,26–0,34, e com 72 h para 0,37–0,47. Uma falha que dura mais que algumas horas entra na média e no desvio móveis, o limiar sobe junto e a janela volta a parecer normal. Os incidentes longos e sustentados (16, 21, 26, 28a, 29a) são justamente os que se perdem.
- **Mediana e MAD móveis são piores ainda** (0,26–0,43, com 18 a 53 falsos positivos). O MAD de poucos dias é pequeno e instável, e o limiar fica colado nos dados.
- **O piso no desvio do treino é o que faz o limiar móvel funcionar.** Com referência de 14 dias e piso, k = 3 dá 18/13/11 (F1 0,60) e k = 4,5 dá 14/5/15 (precisão 0,74, F1 0,58). Os incidentes ganhos são 7b, 8 e 17, de cenários em que o nível normal da predição se afastou do treino: o que o limiar móvel corrige ali é a *média* do baseline, não o desvio.
- **Fixo E móvel** corta falsos positivos sem custo de recall: com 14 dias e k = 3, 16 incidentes com 9 falsos positivos em vez de 12 (F1 0,59). **Fixo OU móvel** quase não muda o resultado (0,55–0,59).

### Baseline adaptativo com congelamento

Uma variação comum evita que a falha contamine a referência: média e variância exponenciais que andam enquanto a janela está normal e **congelam** (ou andam bem mais devagar) quando ela entra em nível de alarme.

In [ ]:
def adaptive_rule(half_life_days: float, k: float, leak: float = 0.0, floor: float = 0.5):
    """Baseline que acompanha o nível normal: média e variância exponenciais, partindo das do treino.

    A cada janela (passo de 1 h) o baseline anda em direção às leituras novas. Com a janela em nível
    de alarme (severidade >= 1) ele anda `leak` vezes mais devagar; `leak = 0` congela o baseline.
    """
    weight = 1 - np.exp(-np.log(2) / (24 * half_life_days))

    def make(arrays: ScenarioArrays) -> Rule:
        fit_center, fit_scale = fit_mean_std(arrays)
        center, variance, seen = fit_center.copy(), fit_scale**2, arrays.n_fit

        def rule(span: Span) -> tuple:
            nonlocal center, variance, seen
            scale = np.maximum(np.sqrt(variance), floor * fit_scale)
            abnormal, severity = share_rule((arrays.norms[span.start : span.stop] - center) / scale, k)
            new = arrays.norms[max(seen, span.start) : span.stop]
            seen = span.stop
            if len(new):
                step = weight * (leak if severity >= 1 else 1.0)
                center = center + step * (new.mean(axis=0) - center)
                variance = variance + step * (((new - center) ** 2).mean(axis=0) - variance)
            return abnormal, severity
        return rule
    return make


adaptive_tests = compare({
    (f"meia-vida de {half_life} d", f"k = {k:g}", "congela" if leak == 0 else f"anda {leak:g}x mais devagar"):
        raise_alerts(score_windows(adaptive_rule(half_life, k, leak)))
    for half_life in (3, 7, 14) for k in (3, 4.5) for leak in (0.0, 0.1)
})
adaptive_tests.index = pd.MultiIndex.from_tuples(adaptive_tests.index, names=["meia-vida", "limite", "em alarme"])
display(adaptive_tests)

**Congelar não resolve.** Os resultados ficam entre 0,46 e 0,58, na mesma faixa do limiar móvel de 14 dias e sem superar o limiar fixo. Com k = 4,5 o baseline adaptativo é preciso (3 a 9 falsos positivos) mas perde incidentes que sobem devagar, como o 21 e o 26: antes de a janela chegar ao nível de alarme o baseline já acompanhou a subida. Com k = 3 ele recupera o recall ao custo de 16 a 25 falsos positivos.

### Os dois lados do limiar móvel

Os dois cenários abaixo mostram a severidade da janela com o limiar fixo, com a referência móvel de 24 h e com a de 14 dias com piso.

In [ ]:
MOVING_EXAMPLES = {
    "Limiar fixo (k = 4,5)": (BASE_OUTPUTS, NEUTRAL),
    "Móvel de 24 h (k = 4,5)": (MOVING_OUTPUTS[("Média e desvio móveis", "24 h, k = 4.5")], "#eb6834"),
    "Móvel de 14 d com piso (k = 3)": (
        MOVING_OUTPUTS[("Desvio móvel com piso no desvio do treino", "14 d, k = 3")], "#2a78d6"),
}


def plot_moving(scenario_id: int) -> None:
    fig = go.Figure()
    for label, (outputs, color) in MOVING_EXAMPLES.items():
        severity = pd.Series({o.timestamp: o.severity for o in outputs[scenario_id]}, dtype=float)
        fig.add_scatter(
            x=severity.index, y=severity.clip(lower=0.05), name=label, mode="lines", connectgaps=False,
            line=dict(color=color, width=1.5), hovertemplate="%{y:.2f}<extra>" + label + "</extra>",
        )
    fig.add_hline(y=1, line=dict(color=NEUTRAL, width=1, dash="dot"))
    for incident in valid_incidents(scenario_id):
        fig.add_vrect(x0=incident.start, x1=incident.end, fillcolor=INCIDENT_FILL, line_width=0, layer="below")
    fig.add_scatter(x=[None], y=[None], mode="markers", name="Incidente rotulado",
                    marker=dict(color=INCIDENT_FILL, symbol="square", size=14))
    style(fig, f"Cenário {scenario_id} — severidade da janela com limiar fixo e com limiar móvel",
          subtitle="Cada linha está em unidades do próprio limite; a janela é anormal acima de 1 (linha pontilhada)")
    fig.update_yaxes(type="log", tickvals=LOG_TICKS, title_text="Severidade")
    fig.update_xaxes(tickformat="%d/%m %H:%M", hoverformat="%d/%m/%Y %H:%M", title_text="Tempo (UTC)")
    fig.show()


for scenario_id in (16, 17):
    plot_moving(scenario_id)

- **Cenário 16** (incidente de 100 h, subida lenta e moderada). Com o limiar fixo, 13 das 100 janelas do incidente são anormais e o alerta sai. Com a referência de 24 h nenhuma janela chega ao limite (severidade máxima de 0,61): a subida entra na média móvel em poucas horas. A referência de 14 dias com piso se comporta quase como o limiar fixo (12 janelas anormais).
- **Cenário 17** (incidente de 64 h). O nível sobe, mas fica abaixo do limiar fixo o tempo todo (severidade máxima de 0,78), porque o normal deste ativo no período de predição está abaixo da média do treino. Medido contra os 14 dias anteriores, o mesmo trecho passa do limite em 40 das 64 janelas. A referência de 24 h volta a absorver a subida (máxima de 0,69).

**Resumo da seção.** Um limiar dinâmico de janela curta, que é como média móvel + desvio móvel costuma ser usado, é a escolha errada para este problema: os incidentes duram de horas a dias e o limiar os absorve. A única variante que se sustenta é a de horizonte longo (14 dias) com piso no desvio do treino, que funciona como uma correção lenta do *nível* do baseline. O ganho é de +0,04 de F1 no melhor caso, dentro do ruído (seção 15), e depende de dois valores escolhidos nestes mesmos cenários (14 dias e k = 3).

## 15. Onde estão os erros que sobram, e o que mais pode ajudar

As seções 12 a 14 mexeram no detector e nenhuma mudança se destacou. Antes de testar outras abordagens, vale olhar cada erro do baseline ajustado: os 13 incidentes não encontrados e os 12 falsos positivos.

- **Primeira tabela** — uma linha por incidente: quantas janelas caem dentro dele e quantas são anormais, o nível da janela (percentil 80 do z, na pior norma) dentro do incidente e nas 48 h anteriores, e se já havia um evento aberto quando o incidente começou.
- **Segunda tabela** — os falsos positivos e a distância até o incidente rotulado mais próximo, antes e depois.

In [ ]:
def trace_engine(outputs: list) -> pd.DataFrame:
    """Passa as janelas de um cenário pelo motor de alertas e guarda o que ele viu e decidiu em cada uma."""
    engine, rows = WindowCountAlertEngine(params=WINDOW_ALERT_PARAMS), []
    for output in outputs:
        event_open = engine._event_open  # estado interno do motor, lido só para este diagnóstico
        decision = engine.predict(output)
        rows.append({
            "timestamp": output.timestamp, "abnormal": output.anomaly_status, "kind": decision.kind,
            "event_open": event_open,
            # Nível da janela, em desvios-padrão do treino.
            "z": np.nan if output.severity is None else output.severity * K_ADJUSTED,
        })
    return pd.DataFrame(rows).set_index("timestamp")


TRACES = {scenario_id: trace_engine(outputs) for scenario_id, outputs in BASE_OUTPUTS.items()}
DAY = pd.Timedelta(hours=24)


def hours_between(earlier: datetime, later: datetime) -> float:
    return (later - earlier) / pd.Timedelta(hours=1)


def incident_row(trace: pd.DataFrame, incident) -> dict:
    if incident.start >= incident.end:
        return {"motivo": "rótulo termina antes de começar"}
    inside = trace[(trace.index >= incident.start) & (trace.index <= incident.end)]
    before = trace[(trace.index >= incident.start - 2 * DAY) & (trace.index < incident.start)]
    alerts = inside.loc[inside["kind"] != "none", "kind"]
    open_at_start = bool(inside["event_open"].iloc[0])
    if len(alerts):
        reason = ""
    elif open_at_start:
        reason = "evento já aberto antes do rótulo"
    else:
        reason = f"menos de {WINDOW_ALERT_PARAMS.onset_windows} janelas anormais seguidas"
    return {
        "horas": hours_between(incident.start, incident.end),
        "janelas": len(inside), "janelas_anormais": int(inside["abnormal"].sum()),
        "z_máx": inside["z"].max(), "z_mediano": inside["z"].median(), "z_mediano_48h_antes": before["z"].median(),
        "evento_aberto_no_início": open_at_start,
        "alertas": ", ".join(ALERT_KINDS[kind] for kind in alerts), "motivo": reason,
    }


incident_diagnosis = pd.DataFrame.from_dict({
    f"{scenario_id}{'ab'[position] if len(incidents) > 1 else ''}": incident_row(TRACES[scenario_id], incident)
    for scenario_id, (_, _, incidents) in SCENARIOS.items() for position, incident in enumerate(incidents)
}, orient="index").rename_axis("incidente")
incident_diagnosis = incident_diagnosis.loc[sorted(incident_diagnosis.index, key=incident_order)]
incident_diagnosis = incident_diagnosis.astype({"janelas": "Int64", "janelas_anormais": "Int64"})
display(incident_diagnosis.round(1).fillna({"evento_aberto_no_início": "", "alertas": ""}))
print("Incidentes não encontrados, por motivo:")
display(incident_diagnosis.loc[incident_diagnosis["motivo"] != "", "motivo"].value_counts().to_frame("incidentes"))


def false_positive_rows(scenario_id: int) -> list:
    trace, incidents = TRACES[scenario_id], valid_incidents(scenario_id)
    rows = []
    for timestamp, window in trace[trace["kind"] != "none"].iterrows():
        if any(incident.start <= timestamp <= incident.end for incident in incidents):
            continue
        until_next = [hours_between(timestamp, incident.start) for incident in incidents if incident.start > timestamp]
        since_last = [hours_between(incident.end, timestamp) for incident in incidents if incident.end < timestamp]
        rows.append({
            "cenário": scenario_id, "alerta": f"{timestamp:%d/%m %H:%M}", "tipo": ALERT_KINDS[window["kind"]],
            "z": window["z"],
            "horas_até_o_próximo_incidente": min(until_next, default=np.nan),
            "horas_desde_o_incidente_anterior": min(since_last, default=np.nan),
        })
    return rows


false_positives = pd.DataFrame([row for scenario_id in SCENARIOS for row in false_positive_rows(scenario_id)])
display(false_positives.round(1))

**Os 13 incidentes não encontrados têm três causas, e só uma delas é do detector.**

| Causa | Incidentes | O que ajudaria |
|---|---|---|
| Rótulo termina antes de começar | 7a | nada: o recall máximo é 28/29 |
| Menos de 3 janelas anormais seguidas | 6a, 7b, 11, 13, 14, 17, 27a | um detector mais sensível, mas em 11, 13, 14 e 27a o nível não passa de 3 desvios |
| Evento já aberto antes do rótulo | 4a, 8, 27b, 28b, 29b | nada no detector: o alerta saiu, só que antes da janela rotulada |

- **Cinco incidentes são perdidos por alertar cedo.** Nos cenários 4, 8, 27, 28 e 29 a vibração sobe horas ou dias antes do início do rótulo (nível mediano de 1,9 a 8,6 desvios nas 48 h anteriores). O alerta sai nesse momento e é contado como falso positivo; dentro da janela rotulada o evento continua aberto e a severidade não dobra. Cada um desses casos custa **um falso positivo e um incidente perdido** ao mesmo tempo.
- **Sete dos 12 falsos positivos estão colados num rótulo:** cinco saem de 5 a 28 h antes de um incidente (cenários 2, 3, 4, 10 e 28) e dois até cerca de 1 h depois do fim (cenários 3 e 6). Os outros cinco estão a mais de 2 dias de qualquer rótulo.
- **Quatro incidentes não têm desvio para cima** que um detector de vibração alta consiga ver (11, 13, 14 e 27a, com nível máximo de 1,7 a 3,0 desvios). Os outros três da mesma linha (6a, 7b e 17) têm algum sinal, fraco ou intermitente.

O gargalo é menos o detector e mais a combinação de "um alerta por evento" com rótulos que marcam uma janela dentro de um período anormal mais longo. As abordagens abaixo atacam esses pontos, todas dentro do que pode ser alterado (`anomaly_model.py`, `alert_engine.py`, `interface.py` e os YAML):

| Família | Ideia | O que ataca |
|---|---|---|
| **Regra em duas escalas** | Além das últimas 2 h, julgar um trecho mais longo da janela de 12 h (6 h ou 12 h), com limite menor e fração maior | desvios pequenos e sustentados (14, 17) |
| **Motor de alertas** | Variar um parâmetro por vez no baseline ajustado (a seção 10 fez isso para o PCA) | confirmação, re-alerta e fechamento do evento |
| **Ajustado junto com o PCA** | Janela anormal se as duas abordagens concordarem (E) ou se alguma disparar (OU) | precisão (E) ou recall (OU) |
| **Um evento por sinal** | Um motor de alertas para cada sinal (velocidade, aceleração, PCA, limiar móvel); cada um abre e fecha o próprio evento | incidentes escondidos atrás de um evento já aberto por outro sinal |

In [ ]:
def multiscale_rule(long_hours: float, long_k: float, long_ratio: float):
    """Regra do baseline nas últimas 2 h OU a mesma regra num trecho mais longo da janela de 12 h.

    O trecho longo usa um limite menor e exige uma fração maior de leituras: procura desvios
    pequenos que se sustentam, que a regra de 2 h não alcança.
    """
    def make(arrays: ScenarioArrays) -> Rule:
        z = norm_z(arrays)

        def rule(span: Span) -> tuple:
            abnormal, severity = share_rule(z[span.start : span.stop], K_ADJUSTED)
            long_from = arrays.times[span.stop - 1] - hours(long_hours)
            long_start = max(int(np.searchsorted(arrays.times, long_from, "right")), span.window_start)
            if span.stop - long_start >= MIN_SAMPLES * long_hours / SPAN_HOURS:
                long_abnormal, long_severity = share_rule(z[long_start : span.stop], long_k, long_ratio)
                abnormal, severity = abnormal or long_abnormal, max(severity, long_severity)
            return abnormal, severity
        return rule
    return make


def merge_outputs(first: dict, second: dict, require_both: bool) -> dict:
    """Junta duas abordagens janela a janela: anormal se as duas estiverem (`require_both`) ou se alguma estiver."""
    merged = {}
    for scenario_id in first:
        merged[scenario_id] = []
        for a, b in zip(first[scenario_id], second[scenario_id]):
            if a.severity is None or b.severity is None:
                merged[scenario_id].append(a)
                continue
            if require_both:
                abnormal, severity = a.anomaly_status and b.anomaly_status, min(a.severity, b.severity)
            else:
                abnormal, severity = a.anomaly_status or b.anomaly_status, max(a.severity, b.severity)
            merged[scenario_id].append(PredictOutput(
                anomaly_status=abnormal, timestamp=a.timestamp, severity=severity, n_valid=a.n_valid))
    return merged


def separate_events(*signals: dict, alert_params=WINDOW_ALERT_PARAMS) -> dict:
    """Um motor de alertas por sinal: cada sinal abre, mantém e fecha o próprio evento; alerta quando algum alerta."""
    decisions = {}
    for scenario_id in SCENARIOS:
        engines = [WindowCountAlertEngine(params=alert_params) for _ in signals]
        decisions[scenario_id] = []
        for outputs in zip(*(signal[scenario_id] for signal in signals)):
            raised = [engine.predict(output) for engine, output in zip(engines, outputs)]
            decisions[scenario_id].append(next((decision for decision in raised if decision.alert), raised[0]))
    return decisions


def with_alert_params(outputs: dict, **update) -> dict:
    alert_params = WINDOW_ALERT_PARAMS.model_copy(update=update)
    return raise_alerts(outputs, lambda: WindowCountAlertEngine(params=alert_params))


PCA_WINDOWS = main_outputs("pca")
VELOCITY_OUTPUTS = score_windows(reading_rule(represent=lambda arrays: arrays.norms[:, :1]))
ACCELERATION_OUTPUTS = score_windows(reading_rule(represent=lambda arrays: arrays.norms[:, 1:]))
MOVING_14D = MOVING_OUTPUTS[("Desvio móvel com piso no desvio do treino", "14 d, k = 3")]

ALTERNATIVES = {
    "Regra em duas escalas (2 h OU trecho longo)": {
        f"{long_hours} h, k = {long_k:g}, fração {long_ratio:.0%}": raise_alerts(score_windows(
            multiscale_rule(long_hours, long_k, long_ratio)))
        for long_hours in (6, 12) for long_k in (2.5, 3, 3.5) for long_ratio in (0.5, 0.7)
    },
    "Motor de alertas, um parâmetro por vez": {
        f"{parameter} = {value:g}": with_alert_params(BASE_OUTPUTS, **{parameter: value})
        for parameter, values in [("onset_windows", (2, 4, 6)), ("escalation_ratio", (1.5, 3.0)),
                                  ("escalation_windows", (1, 3)), ("recovery_windows", (6, 12, 48, 72))]
        for value in values
    },
    "Baseline ajustado junto com o PCA": {
        "PCA sozinho": raise_alerts(PCA_WINDOWS),
        "Ajustado E PCA": raise_alerts(merge_outputs(BASE_OUTPUTS, PCA_WINDOWS, require_both=True)),
        "Ajustado OU PCA": raise_alerts(merge_outputs(BASE_OUTPUTS, PCA_WINDOWS, require_both=False)),
    },
    "Um evento por sinal": {
        "Velocidade | aceleração": separate_events(VELOCITY_OUTPUTS, ACCELERATION_OUTPUTS),
        "Limiar fixo | móvel de 14 d": separate_events(BASE_OUTPUTS, MOVING_14D),
        "Ajustado | PCA": separate_events(BASE_OUTPUTS, PCA_WINDOWS),
        "Velocidade | aceleração | PCA": separate_events(VELOCITY_OUTPUTS, ACCELERATION_OUTPUTS, PCA_WINDOWS),
    },
}
alternative_tests = pd.concat({family: compare(variants) for family, variants in ALTERNATIVES.items()},
                              names=["família", "variante"])
display(alternative_tests)

- **Duas escalas: não ajuda** (0,52–0,56). O trecho longo encontra 17 e 28b, mas abre eventos mais cedo em outros cenários e perde o 26; o saldo é zero ou negativo.
- **Motor de alertas: os valores atuais estão certos, com uma exceção pequena.** Confirmar em 2 janelas dobra os falsos positivos (24), e confirmar em 4 ou 6 perde os incidentes curtos. Fechar o evento mais cedo (6 ou 12 janelas normais) gera alertas repetidos do mesmo período anormal. Esperar **48 janelas normais** em vez de 24 tira 2 falsos positivos sem perder nenhum incidente (16/10/13, F1 0,58).
- **Ajustado E PCA é a combinação mais precisa:** 16 incidentes com 9 falsos positivos (precisão 0,64, F1 0,59). Exigir a concordância dos dois detectores evita abrir eventos em desvios que só um deles vê. **Ajustado OU PCA** piora (0,48), porque qualquer um dos dois abre o evento cedo.
- **Um evento por sinal mostra o teto de recall.** Com um motor para o baseline ajustado e outro para o PCA, 23 dos 28 incidentes válidos são encontrados (recall 0,79), incluindo três dos cinco perdidos por um evento já aberto (8, 28b e 29b). O preço são 27 falsos positivos, e o F1 fica em 0,58. Os dois detectores veem incidentes diferentes, mas cada sinal a mais traz os próprios alertas antecipados.

### As melhores candidatas, com o ruído medido

Para cada candidata, o F1 é recalculado em 4000 reamostragens dos cenários. `ganho` é a diferença para o baseline ajustado na mesma reamostragem, e `prob_de_ser_melhor` é a fração das reamostragens em que a candidata tem F1 maior que ele.

In [ ]:
CANDIDATES = {
    "Baseline ajustado (referência)": BASE_DECISIONS,
    "+ média da janela (limite 6)": STAT_DECISIONS[("com a regra do baseline", "média", 6)],
    "recovery_windows = 48": ALTERNATIVES["Motor de alertas, um parâmetro por vez"]["recovery_windows = 48"],
    "Móvel de 14 d com piso, k = 3": raise_alerts(MOVING_14D),
    "Móvel de 14 d com piso, k = 4,5": raise_alerts(
        MOVING_OUTPUTS[("Desvio móvel com piso no desvio do treino", "14 d, k = 4.5")]),
    "Fixo E móvel de 14 d (k = 3)": raise_alerts(MOVING_OUTPUTS[("Fixo E móvel", "14 d, k = 3")]),
    "Ajustado E PCA": ALTERNATIVES["Baseline ajustado junto com o PCA"]["Ajustado E PCA"],
    "PCA sozinho": ALTERNATIVES["Baseline ajustado junto com o PCA"]["PCA sozinho"],
    "Um evento por sinal: ajustado | PCA": ALTERNATIVES["Um evento por sinal"]["Ajustado | PCA"],
}


def ranking_row(decisions: dict) -> dict:
    f1 = bootstrap_f1(decisions)
    gain = f1 - BASE_F1
    return {
        **global_metrics(decisions),
        "f1_p5": np.quantile(f1, 0.05), "f1_p95": np.quantile(f1, 0.95),
        "ganho_p5": np.quantile(gain, 0.05), "ganho_p95": np.quantile(gain, 0.95),
        "prob_de_ser_melhor": (gain > 0).mean(),
    }


ranking = pd.DataFrame.from_dict(
    {name: ranking_row(decisions) for name, decisions in CANDIDATES.items()}, orient="index")
display(ranking.round(2))

fig = go.Figure(go.Scatter(
    x=ranking["f1"], y=ranking.index, mode="markers", marker=dict(color="#2a78d6", size=11),
    error_x=dict(type="data", symmetric=False, array=ranking["f1_p95"] - ranking["f1"],
                 arrayminus=ranking["f1"] - ranking["f1_p5"], color="#8a8985", thickness=1.5, width=6),
    hovertemplate="%{y}: F1 %{x:.2f}<extra></extra>",
))
fig.add_vline(x=ranking["f1"].iloc[0], line=dict(color=NEUTRAL, width=1, dash="dot"))
style(fig, "F1 de cada candidata, com o intervalo de 90% do bootstrap sobre os cenários", height=460, legend=False,
      subtitle="A linha pontilhada marca o baseline ajustado")
fig.update_layout(hovermode="closest", margin=dict(l=280))
fig.update_xaxes(title_text="F1 global", range=[0.3, 0.85])
fig.update_yaxes(autorange="reversed")
fig.show()

**Nenhuma candidata se separa do baseline ajustado.** Os ganhos vão de 0 a +0,04 de F1, e em todas o intervalo de 90% do ganho inclui zero. A probabilidade de ser melhor fica entre 0,50 e 0,74, longe do que permitiria afirmar uma melhora com 29 cenários.

**O que parece promissor, em ordem de confiança:**

1. **`recovery_windows: 48`.** A única mudança que não piora em nenhuma reamostragem (ganho de 0,00 a +0,06): tira 2 falsos positivos e custa um parâmetro. Medida aqui no baseline ajustado; a seção 10 tem a mesma varredura para o PCA.
2. **Exigir concordância entre o baseline ajustado e o PCA.** Três falsos positivos a menos (9 contra 12) para o mesmo número de incidentes encontrados (16), mantendo as contribuições por variável do PCA.
3. **Referência móvel de 14 dias com piso no desvio do treino.** O maior F1 (0,60) e a única que encontra incidentes novos (7b, 8 e 17) quase sem custo, mas com o intervalo mais largo (−0,06 a +0,15) e com estado guardado no modelo. É a que ataca a causa apontada na seção 13, o baseline do treino que envelhece, e por isso a que mais vale revalidar com mais dados.
4. **Um evento por sinal.** O caminho para levar o recall acima de ~0,62, se o custo em alertas for aceitável para o produto.

**O que não vale insistir:** mais estatísticas da janela (seção 12), limites por ativo calculados no treino (seção 13), limiar móvel de horas ou poucos dias (seção 14), regra em duas escalas e confirmação mais curta.

**O que os dados não permitem testar:** fator de crista, curtose e bandas de frequência do sinal bruto (só há RMS a cada 10 min), e variáveis de contexto de operação como carga e rotação, que explicariam parte das mudanças de nível que hoje viram falso positivo.

**O limite da métrica.** Dos 29 incidentes, 1 tem rótulo inválido, 4 não têm desvio para cima e 5 começam depois de o alerta já ter saído. Com um único estado de evento, um detector de nível fica perto de 19 incidentes encontrados; separar os eventos por sinal leva a 23. Daí em diante, o que separa as abordagens é quantos falsos positivos cada uma paga por esses incidentes.

## 16. Abordagem final: nível sustentado contra o baseline do ativo

As seções 4 a 15 deixaram lições claras sobre o baseline ajustado e o PCA. A abordagem final é a resposta direta a elas:

| O que as seções anteriores mostraram | Decisão na abordagem final |
|---|---|
| A informação útil está no **nível** da vibração; estatísticas de forma ou de oscilação da janela não têm sinal (seções 11 e 12). | Uma severidade por janela, construída só a partir do nível. |
| O mesmo limite em desvios-padrão não significa a mesma coisa em ativos diferentes: a maior leitura do treino fica de 1,3 a 43,8 desvios da média (seção 13). | O baseline é o **percentil 99 das leituras com a máquina rodando** no treino, e a severidade é a **razão** nível ÷ baseline. Um limite de 1,5× quer dizer "50% acima do topo do que esta máquina faz normalmente" em qualquer ativo, sem supor distribuição nenhuma. É a linguagem que analistas de vibração já usam. |
| Surtos curtos e ruído abrem eventos cedo, e o evento aberto antes do rótulo custa um falso positivo e um incidente (seção 15). | O nível **sustentado** é a k-ésima maior leitura das últimas 3 h, com k = 35% do horizonte nominal (cerca de 1 h de evidência). Um pico isolado não move esse nível, e uma janela com poucas leituras rodando tem severidade 0. |
| No PCA, a confirmação em 3 janelas compensava um detector ruidoso, mas contar janelas faz o motor depender do passo do pipeline e das janelas repetidas dentro de um buraco (seções 9, 10 e 15). | A evidência sustentada fica dentro do modelo, então o motor alerta na **primeira** janela anormal, re-alerta quando a severidade chega a **2×** a do último alerta, e fecha o evento depois de **24 h sem janela anormal**, medidas pelos timestamps e não por contagem de chamadas. |
| Buracos de telemetria encerram o janelamento (seção 1). | Janelas de 12 h com passo de 30 min; o modelo julga só as últimas 3 h de cada uma. |

Implementação: `classes/anomaly_model.py` (`AnomalyModel`), `classes/alert_engine.py` (`AlertEngine`) e os três YAMLs em `hyperparameters/`. A primeira célula confirma que o notebook reproduz o `main.py` exatamente.

In [ ]:
FINAL = "sustained"
FINAL_OUTPUTS = main_outputs(FINAL)
FINAL_DECISIONS = main_decisions(FINAL)
FINAL_WINDOWS = WINDOWS[MAIN_WINDOWING[FINAL]]


def final_engine(alert_update: dict | None = None):
    params = ALERT_PARAMS.model_copy(update=alert_update or {})
    return lambda: AlertEngine(params=params)


def score_final(model_update: dict | None = None, model_class=AnomalyModel, windows: dict | None = None) -> dict:
    """Saída do modelo final em cada janela, com hiperparâmetros do modelo trocados por `model_update`."""
    params = MODEL_PARAMS.model_copy(update=model_update or {})
    windows = windows or FINAL_WINDOWS
    outputs = {}
    for scenario_id, (fit, _, _) in SCENARIOS.items():
        model = model_class(params=params)
        model.fit(fit)
        outputs[scenario_id] = [model.predict(window) for window in windows[scenario_id]]
    return outputs


def with_threshold(outputs: dict, threshold: float) -> dict:
    # A severidade não depende do limite, então variar o limite não exige pontuar as janelas de novo.
    return {i: [o.model_copy(update={"anomaly_status": o.severity >= threshold}) for o in scenario_outputs]
            for i, scenario_outputs in outputs.items()}


def final_metrics(outputs: dict | None = None, alert_update: dict | None = None) -> dict:
    return global_metrics(raise_alerts(outputs or FINAL_OUTPUTS, final_engine(alert_update)))


# O notebook reproduz exatamente o que o main.py calcula (mesmo modelo, mesmas janelas, mesmo motor).
from utils.utils import run_experiment

pipeline_runs = {i: run_experiment(*SCENARIOS[i])[0] for i in SCENARIOS}
pipeline_totals = aggregate_results(pipeline_runs)
assert (pipeline_totals["TP_total"], pipeline_totals["FP_total"], pipeline_totals["FN_total"]) == tuple(
    global_metrics(FINAL_DECISIONS)[c] for c in ("TP", "FP", "FN")), "o notebook deveria reproduzir o main.py"
assert final_metrics() == global_metrics(FINAL_DECISIONS)
print("main.py:", {k: round(v, 3) for k, v in pipeline_totals.items()})

### Placar e alertas da abordagem final

Cada alerta traz a mensagem do motor: a medida que puxou a severidade, o nível em múltiplos do baseline e, numa escalada, o nível do alerta anterior.

In [ ]:
final_alerts = pd.DataFrame([
    {"cenário": i, "timestamp": d.timestamp, "tipo": ALERT_KINDS[d.kind],
     "dentro_do_incidente": any(inc.start <= d.timestamp <= inc.end for inc in valid_incidents(i)),
     "mensagem": d.message}
    for i in SCENARIOS for d in FINAL_DECISIONS[i] if d.alert
])
pd.set_option("display.max_colwidth", 120)
display(final_alerts)
print(f"Janelas pontuadas: {sum(len(v) for v in FINAL_OUTPUTS.values())}; "
      f"janelas anormais: {sum(o.anomaly_status for v in FINAL_OUTPUTS.values() for o in v)}; "
      f"alertas: {len(final_alerts)} ({(final_alerts['tipo'] == ALERT_KINDS['escalation']).sum()} escaladas).")
silent = [i for i in SCENARIOS if not SCENARIOS[i][2] and not any(d.alert for d in FINAL_DECISIONS[i])]
print(f"Cenários sem incidente rotulado: {[i for i in SCENARIOS if not SCENARIOS[i][2]]}; sem nenhum alerta: {silent}")

### Severidade ao longo do tempo

Uma figura por cenário: a severidade de cada janela (nível sustentado ÷ baseline, na pior das duas normas), o limite de 1,5 (tracejado), o topo do normal do treino (1,0, pontilhado), os incidentes rotulados e os alertas (▲ novo evento, ◆ escalada). O hover mostra qual norma puxou a severidade.

In [ ]:
DRIVER_LABELS = {"velocity": "velocidade", "acceleration": "aceleração"}


def plot_final(scenario_id: int) -> None:
    frame = pd.DataFrame([
        {"timestamp": o.timestamp, "severity": o.severity, "driver": DRIVER_LABELS.get(o.driver, "—"), "kind": d.kind}
        for o, d in zip(FINAL_OUTPUTS[scenario_id], FINAL_DECISIONS[scenario_id])
    ]).set_index("timestamp")
    fig = go.Figure()
    fig.add_scatter(x=frame.index, y=frame["severity"].clip(lower=0.05), mode="lines", name="Severidade",
                    line=dict(color=APPROACH_COLORS[FINAL], width=1.5), customdata=frame["driver"],
                    hovertemplate="%{x|%d/%m %H:%M} — %{y:.2f}x (%{customdata})<extra></extra>")
    for kind, label, symbol in [("onset", "Alerta: novo evento", "triangle-up"), ("escalation", "Alerta: escalada", "diamond")]:
        alerts = frame[frame["kind"] == kind]
        fig.add_scatter(x=alerts.index, y=alerts["severity"], mode="markers", name=label,
                        marker=dict(color=INK, size=11, symbol=symbol, line=dict(color="white", width=2)),
                        hovertemplate="%{x|%d/%m %H:%M} — %{y:.2f}x<extra>" + label + "</extra>")
    fig.add_hline(y=MODEL_PARAMS.severity_threshold, line=dict(color=INK, width=1, dash="dash"))
    fig.add_hline(y=1.0, line=dict(color=NEUTRAL, width=1, dash="dot"))
    for incident in valid_incidents(scenario_id):
        fig.add_vrect(x0=incident.start, x1=incident.end, fillcolor=INCIDENT_FILL, line_width=0, layer="below")
    fig.add_scatter(x=[None], y=[None], mode="markers", name="Incidente rotulado",
                    marker=dict(color=INCIDENT_FILL, symbol="square", size=14))
    counts = scenario_metrics({scenario_id: FINAL_DECISIONS[scenario_id]}).iloc[0]
    style(fig, f"Cenário {scenario_id} — {APPROACHES[FINAL]}", height=380,
          subtitle=f"{counts['incidentes']} incidente(s) · {counts['alertas']} alerta(s) · "
                   f"TP {counts['TP']}, FP {counts['FP']}, FN {counts['FN']} · tracejado = limite de "
                   f"{MODEL_PARAMS.severity_threshold:g}x, pontilhado = topo do normal")
    fig.update_layout(hovermode="closest")
    fig.update_yaxes(type="log", tickvals=LOG_TICKS, title_text="Nível ÷ baseline")
    fig.update_xaxes(tickformat="%d/%m", hoverformat="%d/%m/%Y %H:%M", title_text="Tempo (UTC)")
    fig.show()


for scenario_id in SCENARIOS:
    plot_final(scenario_id)

### Quais medidas o modelo precisa?

O modelo final usa as normas da velocidade e da aceleração. As subclasses abaixo trocam só `_featuring`: uma norma de cada vez, ou os seis eixos, cada um contra o próprio baseline.

In [ ]:
class SingleNorm(AnomalyModel):
    """O modelo final restrito a uma das duas normas."""

    norm: str

    def _featuring(self, samples):
        timestamps, running, features = super()._featuring(samples)
        return timestamps, running, {self.norm: features[self.norm]}


class VelocityOnly(SingleNorm):
    norm = "velocity"


class AccelerationOnly(SingleNorm):
    norm = "acceleration"


class PerAxis(AnomalyModel):
    """Cada eixo contra o próprio baseline, em vez de uma norma por sensor."""

    def _featuring(self, samples):
        timestamps, running, _ = super()._featuring(samples)
        ordered = sorted(samples.data, key=lambda p: p.timestamp)
        return timestamps, running, {axis: np.array([getattr(p, axis) for p in ordered]) for axis in FEATURE_NAMES}


per_axis_outputs = score_final(model_class=PerAxis)
feature_variants = {
    "Normas da velocidade e da aceleração (final)": FINAL_OUTPUTS,
    "Só a norma da velocidade": score_final(model_class=VelocityOnly),
    "Só a norma da aceleração": score_final(model_class=AccelerationOnly),
    "Seis eixos, limite 1,5": per_axis_outputs,
    "Seis eixos, limite 1,8": with_threshold(per_axis_outputs, 1.8),
    "Seis eixos, limite 2,0": with_threshold(per_axis_outputs, 2.0),
}
feature_table = pd.DataFrame({label: final_metrics(outputs) for label, outputs in feature_variants.items()}).T
feature_table[["TP", "FP", "FN"]] = feature_table[["TP", "FP", "FN"]].astype(int)
display(feature_table.round(3))

### Sensibilidade a cada hiperparâmetro

Um hiperparâmetro varia por vez, os outros ficam nos valores de `hyperparameters/*.yaml`. `1e9` em `escalation_factor` significa nunca re-alertar; em `rearm_hours`, nunca fechar um evento (a trava do baseline original). Como na seção 10, os rótulos que pontuam são os mesmos usados para escolher os valores: as curvas mostram estabilidade, não desempenho em ativos novos.

In [ ]:
FINAL_SWEEPS = {
    "severity_threshold": ("threshold", [1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 2.0]),
    "window_anomaly_ratio": ("model", [0.15, 0.25, 0.3, 0.35, 0.4, 0.5]),
    "horizon_hours": ("model", [1, 2, 3, 4, 6, 12]),
    "baseline_quantile": ("model", [0.95, 0.975, 0.99, 0.995, 1.0]),
    "escalation_factor": ("alert", [1.25, 1.5, 2.0, 3.0, 1e9]),
    "rearm_hours": ("alert", [0, 6, 12, 24, 48, 72, 1e9]),
}


def sweep_point(parameter: str, owner: str, value: float) -> dict:
    if owner == "threshold":
        return final_metrics(with_threshold(FINAL_OUTPUTS, value))
    if owner == "model":
        return final_metrics(score_final({parameter: value}))
    return final_metrics(alert_update={parameter: value})


final_sweeps = {
    parameter: pd.DataFrame({value: sweep_point(parameter, owner, value) for value in values}).T
    for parameter, (owner, values) in FINAL_SWEEPS.items()
}
CHOSEN = {**MODEL_PARAMS.model_dump(), **ALERT_PARAMS.model_dump()}

fig = make_subplots(rows=2, cols=3, subplot_titles=[f"<b>{p}</b>" for p in FINAL_SWEEPS],
                    vertical_spacing=0.18, horizontal_spacing=0.07)
for position, (parameter, (owner, values)) in enumerate(FINAL_SWEEPS.items()):
    row, col = divmod(position, 3)
    table = final_sweeps[parameter]
    labels = ["nunca" if v >= 1e9 else f"{v:g}" for v in table.index]
    for metric, color in METRIC_COLORS.items():
        fig.add_trace(go.Scatter(
            x=labels, y=table[metric], mode="lines+markers", legendgroup=metric,
            name=METRIC_LABELS[metric], showlegend=position == 0,
            line=dict(color=color, width=2), marker=dict(size=8, line=dict(color="white", width=1)),
            hovertemplate="%{y:.2f}<extra>" + METRIC_LABELS[metric] + "</extra>",
        ), row=row + 1, col=col + 1)
    chosen = [float(v) for v in values].index(float(CHOSEN[parameter]))
    fig.add_vline(x=chosen, line=dict(color=NEUTRAL, width=1, dash="dot"), row=row + 1, col=col + 1)
    fig.update_xaxes(type="category", row=row + 1, col=col + 1)
style(fig, "Abordagem final: precisão, recall e F1 conforme um hiperparâmetro varia", height=640,
      subtitle="A linha pontilhada marca o valor em hyperparameters/*.yaml")
fig.update_layout(hovermode="x unified")
fig.update_yaxes(range=[0, 1.05])
fig.show()

display(pd.concat({p: t[["TP", "FP", "FN", "f1"]] for p, t in final_sweeps.items()}).round(2))

### Tamanho da janela e passo do pipeline

O modelo julga só as últimas `horizon_hours` (3 h) de cada janela, então o tamanho da janela só precisa passar do maior buraco de telemetria (10,1 h), e o passo define de quanto em quanto tempo o ativo é reavaliado.

In [ ]:
WINDOW_CONFIGS = [(2, 1), (6, 0.5), (12, 0.5), (12, 1), (12, 2), (24, 0.5)]  # (tamanho, passo) em horas
window_table = {}
for size, step in WINDOW_CONFIGS:
    windows = slice_windows(PipelineParams(model_window_size_hours=size, window_overlap_hours=size - step))
    label = f"janela de {size:g} h, passo de {60 * step:g} min" + (" (final)" if (size, step) == (12, 0.5) else "")
    window_table[label] = final_metrics(score_final(windows=windows))
window_table = pd.DataFrame(window_table).T
window_table[["TP", "FP", "FN"]] = window_table[["TP", "FP", "FN"]].astype(int)
display(window_table.round(3))

### Quanto do resultado vem de ter escolhido os valores olhando os rótulos?

Validação deixando um cenário de fora: para cada cenário, escolho a melhor configuração de uma grade de 108 pontos (limite, fração do horizonte, horizonte, fator de escalada) usando os outros 28 e aplico ao que ficou de fora. A soma dos cenários deixados de fora é a estimativa honesta do que esperar num ativo novo.

In [ ]:
from itertools import product

LOSO_GRID = {
    "severity_threshold": [1.4, 1.5, 1.6, 1.7],
    "window_anomaly_ratio": [0.25, 0.35, 0.45],
    "horizon_hours": [2, 3, 4],
    "escalation_factor": [1.5, 2.0, 3.0],
}
grid_counts = {}
for ratio, horizon in product(LOSO_GRID["window_anomaly_ratio"], LOSO_GRID["horizon_hours"]):
    outputs = score_final({"window_anomaly_ratio": ratio, "horizon_hours": horizon})
    for threshold in LOSO_GRID["severity_threshold"]:
        thresholded = with_threshold(outputs, threshold)
        for factor in LOSO_GRID["escalation_factor"]:
            decisions = raise_alerts(thresholded, final_engine({"escalation_factor": factor}))
            grid_counts[(threshold, ratio, horizon, factor)] = scenario_counts(decisions)

GRID_CONFIGS = list(product(*LOSO_GRID.values()))  # mesma ordem da grade em todas as dobras
GRID_COUNTS = np.stack([grid_counts[config] for config in GRID_CONFIGS])  # [configuração, cenário, TP/FP/FN]
grid_totals = GRID_COUNTS.sum(axis=1)

held_out, chosen_configs = np.zeros(3, dtype=int), []
for position in range(len(SCENARIOS)):
    best = int(np.argmax(f1_score(grid_totals - GRID_COUNTS[:, position])))
    held_out += GRID_COUNTS[best, position]
    chosen_configs.append(GRID_CONFIGS[best])

final_counts = scenario_counts(FINAL_DECISIONS).sum(axis=0)
usual = max(set(chosen_configs), key=chosen_configs.count)
LOSO = pd.DataFrame({
    "Na amostra (valores do YAML)": dict(zip(("TP", "FP", "FN"), final_counts), f1=f1_score(final_counts)),
    "Fora da amostra (deixando um cenário de fora)": dict(zip(("TP", "FP", "FN"), held_out), f1=f1_score(held_out)),
}).T
display(LOSO.round(3))
in_sample = f1_score(grid_totals)
print(f"Configuração mais escolhida: {dict(zip(LOSO_GRID, usual))} ({chosen_configs.count(usual)} de {len(SCENARIOS)} dobras)")
print(f"F1 na amostra ao longo da grade: melhor {in_sample.max():.2f}, mediana {np.median(in_sample):.2f}, pior {in_sample.min():.2f}")

### Os três requisitos no motor final

Os perfis de manual da seção 9, mais dois que exercitam o fim de um evento, passam pelo motor final (`AlertEngine`) e pelo motor por contagem de janelas usado pelo ajustado e pelo PCA. Uma janela por hora; severidade em múltiplos do baseline (anormal a partir de 1,5).

| Perfil | O que acontece | Alertas desejados |
|---|---|---|
| Platô, degrau, rampa lenta, dente de serra | como na seção 9 | 1 novo evento + 0, 1, 2 e 0 escaladas |
| Intermitente | alterna 2x e 1x por 40 h: o mesmo problema piscando em torno do limite | 1 novo evento |
| Recorrência | 10 h a 2x, 30 h normal, 10 h a 2x: dois eventos separados por mais de um dia | 2 novos eventos |

In [ ]:
FINAL_PROFILES = {
    **PROFILES,
    "Intermitente": [2.0, 1.0] * 20,
    "Recorrência": [2.0] * 10 + [1.0] * 30 + [2.0] * 10,
}
DESIRED = {"Platô": (1, 0), "Degrau": (1, 1), "Rampa lenta": (1, 2), "Dente de serra": (1, 0),
           "Intermitente": (1, 0), "Recorrência": (2, 0)}


def replay(make_engine, severities: list) -> tuple:
    engine, start = make_engine(), datetime(2026, 1, 1, tzinfo=timezone.utc)
    decisions = [engine.predict(PredictOutput(anomaly_status=s >= MODEL_PARAMS.severity_threshold,
                                              timestamp=start + timedelta(hours=k), severity=s))
                 for k, s in enumerate(severities)]
    return sum(d.kind == "onset" for d in decisions), sum(d.kind == "escalation" for d in decisions)


profile_engines = {
    "Motor final (tempo)": lambda: AlertEngine(params=ALERT_PARAMS),
    "Motor por contagem de janelas": lambda: WindowCountAlertEngine(params=WINDOW_ALERT_PARAMS),
}
final_profile_table = pd.DataFrame({
    "desejado": {name: f"{o} + {e}" for name, (o, e) in DESIRED.items()},
    **{label: {name: "{} + {}".format(*replay(make, s)) for name, s in FINAL_PROFILES.items()}
       for label, make in profile_engines.items()},
}).rename_axis("perfil (novos eventos + escaladas)")
display(final_profile_table)

### Atraso de detecção, incidentes perdidos e falsos positivos

`utils.lead_time` não pode ser chamado (passa quatro argumentos a `overlap`), então o atraso é medido aqui: horas entre o início do rótulo e o primeiro alerta dentro dele. Para cada incidente perdido, a tabela dá a maior severidade dentro do rótulo e se um evento já estava aberto quando ele começou.

In [ ]:
def first_alert_delays(decisions: dict) -> pd.Series:
    delays = {}
    for scenario_id, scenario_decisions in decisions.items():
        incidents = SCENARIOS[scenario_id][2]
        for position, incident in enumerate(incidents):
            hits = [d.timestamp for d in scenario_decisions if d.alert and incident.start <= d.timestamp <= incident.end]
            if hits:
                delays[f"{scenario_id}{'ab'[position] if len(incidents) > 1 else ''}"] = hours_between(incident.start, min(hits))
    return pd.Series(delays, name="horas")


final_delays = first_alert_delays(FINAL_DECISIONS)
print("Atraso do primeiro alerta (h): " + ", ".join(
    f"{name} {value:.1f}" for name, value in final_delays.quantile([0.25, 0.5, 0.75]).rename(
        {0.25: "p25", 0.5: "mediana", 0.75: "p75"}).items()) + f", máximo {final_delays.max():.1f}")


def final_trace(scenario_id: int) -> pd.DataFrame:
    engine, rows = AlertEngine(params=ALERT_PARAMS), []
    for output in FINAL_OUTPUTS[scenario_id]:
        engine._close_event_if_recovered(output.timestamp)
        event_open = engine._alerted_severity is not None  # estado interno, lido só para este diagnóstico
        decision = engine.predict(output)
        rows.append({"timestamp": output.timestamp, "severity": output.severity, "kind": decision.kind, "event_open": event_open})
    return pd.DataFrame(rows).set_index("timestamp")


FINAL_TRACES = {scenario_id: final_trace(scenario_id) for scenario_id in SCENARIOS}
missed = []
for scenario_id, (_, _, incidents) in SCENARIOS.items():
    for position, incident in enumerate(incidents):
        label = f"{scenario_id}{'ab'[position] if len(incidents) > 1 else ''}"
        if label in final_delays.index:
            continue
        if incident.start >= incident.end:
            missed.append({"incidente": label, "motivo": "rótulo termina antes de começar"})
            continue
        trace = FINAL_TRACES[scenario_id]
        inside = trace[(trace.index >= incident.start) & (trace.index <= incident.end)]
        peak = inside["severity"].max()
        if bool(inside["event_open"].iloc[0]):
            reason = "evento já aberto antes do rótulo, sem dobrar dentro dele"
        elif peak < MODEL_PARAMS.severity_threshold:
            reason = "severidade abaixo do limite"
        else:
            reason = "outro"
        missed.append({"incidente": label, "horas": hours_between(incident.start, incident.end),
                       "severidade_máxima": peak, "motivo": reason})
missed = pd.DataFrame(missed).set_index("incidente")
display(missed.round(2))
display(missed["motivo"].value_counts().to_frame("incidentes"))

final_fp = []
for scenario_id in SCENARIOS:
    incidents = valid_incidents(scenario_id)
    for d in FINAL_DECISIONS[scenario_id]:
        if d.alert and not any(inc.start <= d.timestamp <= inc.end for inc in incidents):
            final_fp.append({
                "cenário": scenario_id, "alerta": f"{d.timestamp:%d/%m %H:%M}", "tipo": ALERT_KINDS[d.kind],
                "horas_até_o_próximo_incidente": min((hours_between(d.timestamp, inc.start) for inc in incidents
                                                      if inc.start > d.timestamp), default=np.nan),
                "horas_desde_o_incidente_anterior": min((hours_between(inc.end, d.timestamp) for inc in incidents
                                                         if inc.end < d.timestamp), default=np.nan),
                "mensagem": d.message,
            })
display(pd.DataFrame(final_fp).round(1))

### Teste de injeção no modelo final

O mesmo teste da seção 8: nas últimas `horizon_hours` do treino de cada ativo (saudáveis por construção), um canal é multiplicado por um fator, ou os seis de uma vez. Mede quantas janelas o modelo final marca e se a norma que ele aponta (`driver`) é a do canal injetado, lado a lado com o PCA.

In [ ]:
def final_healthy_window(scenario_id: int) -> TimeSeries:
    running = [p for p in SCENARIOS[scenario_id][0].data if p.uptime]
    start = running[-1].timestamp - timedelta(hours=MODEL_PARAMS.horizon_hours)
    return TimeSeries(data=[p for p in running if p.timestamp > start])


def final_injection_test() -> pd.DataFrame:
    rows = []
    for scenario_id in SCENARIOS:
        model, window = MODELS[FINAL][scenario_id], final_healthy_window(scenario_id)
        if model.predict(window).severity == 0:  # poucas leituras rodando: o modelo não julga a janela
            continue
        for factor in INJECTION_FACTORS:
            for channel in FEATURE_NAMES:
                output = model.predict(inject(window, (channel,), factor))
                expected = "velocity" if channel.startswith("vel") else "acceleration"
                rows.append({"abordagem": APPROACHES[FINAL], "fator": factor, "teste": "um canal",
                             "sinalizado": output.anomaly_status, "medida_certa": output.driver == expected})
            output = model.predict(inject(window, FEATURE_NAMES, factor))
            rows.append({"abordagem": APPROACHES[FINAL], "fator": factor, "teste": "todos os canais",
                         "sinalizado": output.anomaly_status, "medida_certa": np.nan})
    return pd.DataFrame(rows)


pca_injections = injection_test("pca").rename(columns={"injetado_em_primeiro": "medida_certa"})
both_injections = pd.concat([pca_injections, final_injection_test()])
injection_summary = both_injections.groupby(["teste", "fator", "abordagem"]).agg(
    casos=("sinalizado", "size"), sinalizados=("sinalizado", "mean"), medida_certa=("medida_certa", "mean"))
display(injection_summary.round(2))
print("No PCA, 'medida_certa' é o canal injetado em primeiro nas contribuições; no modelo final, a norma certa como driver.")

**Leitura da seção 16.**

- **Placar.** 17 incidentes encontrados com 3 falsos positivos (F1 0,69), contra 0,56 do ajustado e do PCA. O ganho vem da precisão: o recall fica entre os dois (17 contra 16 e 18), mas os falsos positivos caem de 12 e 17 para 3. Os sete cenários sem incidente seguem sem nenhum alerta.
- **As duas normas são necessárias.** Só a velocidade encontra 13 incidentes, só a aceleração 8; juntas, 17. Os seis eixos separados encontram os mesmos 17, mas com 13 falsos positivos; no melhor limite próprio (1,8) ficam em 16/6, F1 0,63. A norma troca um pouco de sensibilidade a falhas de um eixo por muito menos ruído.
- **Sensibilidade.** O limite de 1,5× fica no começo de um platô à direita (0,67 em 1,6 e 0,64 em 1,7) e com uma queda à esquerda (0,57 em 1,4), onde o ruído entra. O horizonte de 3 h é um pico (0,60 em 2 h, 0,61 em 4 h): quatro incidentes duram 3 h ou menos. O quantil do baseline tem platô em 0,99–0,995. No motor, o fator de escalada quase não importa (0,63–0,69) e o rearme tem platô de 24 a 72 h. Rearmar a cada janela (0 h) produz 589 falsos positivos, e nunca rearmar (a trava do baseline original) perde dois incidentes.
- **Janela e passo.** 24 h dá o mesmo que 12 h: passado o maior buraco, o tamanho da janela não importa, porque o modelo só julga as últimas 3 h. A janela de 6 h parece melhor (17/1/12), mas é um efeito colateral: com 6 h o janelamento para no buraco de 10,1 h do cenário 28, os últimos 10 dias desse cenário deixam de ser pontuados, e os dois falsos positivos somem junto. O passo de 30 min é melhor que o de 1 h (16/7/13): um evento curto é visto mais cedo e por mais janelas.
- **Fora da amostra.** Escolhendo os quatro parâmetros principais com um cenário de fora, a soma dos cenários deixados de fora dá 14/5/15, F1 0,58. A configuração do YAML é a escolhida em 25 das 29 dobras. 0,58 é a estimativa honesta para um ativo novo; na mesma validação, o baseline ajustado cai para 0,47 (seção 13).
- **Requisitos 1, 2 e 3.** Nos seis perfis de manual, o motor final dá exatamente o número desejado de novos eventos e escaladas. O motor que conta janelas falha no perfil intermitente: um problema que pisca em torno do limite nunca tem três janelas anormais seguidas e nunca é alertado.
- **Atraso.** O primeiro alerta sai uma mediana de 4,1 h depois do início do rótulo (quartis de 1,6 e 10,6 h, máximo de 23,7 h).
- **Os 12 incidentes perdidos.** Um tem o rótulo invertido (7a). Oito ficam abaixo do limite: a maior severidade dentro do rótulo vai de 0,76 a 1,38 (7b, 11, 12, 13, 14, 17, 26 e 27a). Três começam com um evento já aberto e não dobram a severidade dentro do rótulo (27b, 28b e 29b).
- **Os 3 falsos positivos.** Um no cenário 27, 58 h antes de um rótulo, numa máquina visivelmente anormal. Dois no cenário 28, em excursões a 1,5× sem rótulo, a 126 h e a 23 h do segundo incidente.
- **O preço da precisão: falhas de um canal só.** No teste de injeção, um canal multiplicado por 2 é sinalizado em 4% dos casos pelo modelo final e em 69% pelo PCA; com ×3, 30% contra 81%. Os seis canais ×2 são sinalizados em 61% dos casos contra 81%. A abordagem final é deliberadamente menos sensível: um desvio só conta quando a norma inteira passa de 1,5× o topo do normal. Quando sinaliza, aponta a norma certa em 79% a 86% dos casos.

## 17. Que métrica olhar

A métrica do desafio conta incidentes encontrados (pelo menos um alerta dentro do rótulo) e alertas fora de qualquer rótulo. Ela responde a uma pergunta só e deixa de fora coisas que importam para quem recebe o alerta. A tabela abaixo calcula, para as quatro abordagens:

| Métrica | Pergunta que responde | Por que olhar |
|---|---|---|
| Precisão, recall e F1 (oficiais) | Dos alertas, quantos estavam certos? Dos incidentes, quantos foram vistos? | É a régua do desafio; o F1 pesa falsos negativos e falsos positivos igualmente. |
| F2 | O mesmo, com o recall pesando quatro vezes mais. | Em manutenção, perder uma falha costuma custar mais que atender um alarme falso. |
| FP por ativo a cada 30 dias | Quantos alarmes falsos uma máquina gera por mês? | É como a fadiga de alarme é sentida em campo, e não depende de quantos incidentes existem. |
| Ativos sem incidente com alerta | Quantas máquinas saudáveis geraram algum alarme? | O caso mais simples de falso positivo. |
| Alertas por incidente encontrado | Quantas vezes o mesmo evento é anunciado? | Requisito 2. A métrica oficial não penaliza alertas repetidos dentro de um rótulo. |
| Atraso mediano e p75 | Quantas horas depois do início do rótulo sai o primeiro alerta? | Detectar cedo é o objetivo do produto; `utils.lead_time` não pode ser chamado. |
| F1 com tolerância de 24 h | Como fica o placar se alertas até 24 h antes do rótulo contarem como acerto? | Vários rótulos começam depois que a vibração já subiu; um aviso antecipado é útil, mas a métrica oficial o conta como falso positivo. |
| F1 macro | Qual o F1 médio por cenário com incidente? | Evita que cenários com muitos alertas dominem o placar. |
| F1 p5–p95 | Qual o intervalo de 90% do F1, reamostrando os cenários? | Com 29 incidentes, um alerta a mais ou a menos move o F1 em cerca de 0,02. |

In [ ]:
from classes.interface import TrueIncident

TOLERANCE = timedelta(hours=24)


def prediction_days(scenario_id: int) -> float:
    return period_days(SCENARIOS[scenario_id][1])


def f_beta(precision: float, recall: float, beta: float) -> float:
    return (1 + beta**2) * precision * recall / max(beta**2 * precision + recall, EPS)


def with_tolerance(scenario_id: int) -> list:
    # Rótulos válidos começando 24 h antes; o rótulo invertido do cenário 7 fica como está.
    return [TrueIncident(start=inc.start - TOLERANCE, end=inc.end) if inc.start < inc.end else inc
            for inc in SCENARIOS[scenario_id][2]]


def metric_row(decisions: dict) -> dict:
    official = global_metrics(decisions)
    tolerant = aggregate_results({i: evaluate(with_tolerance(i), decisions[i]) for i in decisions})
    delays = first_alert_delays(decisions)
    with_incidents = [i for i in SCENARIOS if SCENARIOS[i][2]]
    macro_f1 = np.mean([evaluate(SCENARIOS[i][2], decisions[i])["f1"] for i in with_incidents])
    no_incident = [i for i in SCENARIOS if not SCENARIOS[i][2]]
    alerts_inside = sum(d.alert and any(inc.start <= d.timestamp <= inc.end for inc in valid_incidents(i))
                        for i in decisions for d in decisions[i])
    low, high = np.quantile(bootstrap_f1(decisions), [0.05, 0.95])
    return {
        **official,
        "F2": f_beta(official["precision"], official["recall"], 2),
        "FP_por_ativo_30d": official["FP"] / sum(prediction_days(i) for i in SCENARIOS) * 30,
        "ativos_sem_incidente_com_alerta": sum(any(d.alert for d in decisions[i]) for i in no_incident),
        "alertas_por_incidente_encontrado": alerts_inside / max(official["TP"], 1),
        "atraso_mediano_h": delays.median(), "atraso_p75_h": delays.quantile(0.75),
        "F1_tolerância_24h": tolerant["f1_global"], "TP_tol": tolerant["TP_total"], "FP_tol": tolerant["FP_total"],
        "F1_macro": macro_f1,
        "F1_p5": low, "F1_p95": high,
    }


METRIC_TABLE = pd.DataFrame({APPROACHES[name]: metric_row(main_decisions(name)) for name in APPROACHES}).T
for column in ("TP", "FP", "FN", "TP_tol", "FP_tol", "ativos_sem_incidente_com_alerta"):
    METRIC_TABLE[column] = METRIC_TABLE[column].astype(int)
display(METRIC_TABLE.T.round(2))
print(f"Monitoramento total: {sum(prediction_days(i) for i in SCENARIOS):.0f} dias-ativo em {len(SCENARIOS)} ativos.")

In [ ]:
# Diferença de F1 entre a abordagem final e as anteriores, nas mesmas reamostragens de cenários (seção 12).
FINAL_F1 = bootstrap_f1(FINAL_DECISIONS)
gains = pd.DataFrame({
    f"final − {APPROACHES[name].lower()}": {
        "ganho_de_F1": FINAL_F1.mean() - reference.mean(),
        "ganho_p5": np.quantile(FINAL_F1 - reference, 0.05),
        "ganho_p95": np.quantile(FINAL_F1 - reference, 0.95),
        "prob_de_ser_melhor": (FINAL_F1 > reference).mean(),
    }
    for name, reference in (("adjusted", BASE_F1), ("pca", bootstrap_f1(main_decisions("pca"))))
}).T
display(gains.round(2))

**Leitura da seção 17.**

- **A abordagem final é a melhor em quase todas as réguas**, não só no F1: 0,16 falso positivo por ativo por mês contra 0,64 a 0,91 das outras, F1 macro de 0,59, F1 com tolerância de 0,73. Nenhum ativo sem incidente gera alerta.
- **Em recall a vantagem é do PCA** (18 contra 17). Com o F2, que favorece o recall, a diferença fica pequena (0,62 contra 0,60). Se o produto aceitasse muito mais alarmes por um incidente a mais, a escolha poderia mudar; a 0,91 alarme falso por máquina por mês contra 0,16, não acho que valha.
- **A tolerância de 24 h mostra o que a métrica oficial esconde.** O PCA passa de 18 para 22 incidentes encontrados: boa parte dos falsos positivos dele são alertas antecipados de incidentes reais. A final passa de 17 para 18, porque os falsos positivos dela não estão colados nos rótulos.
- **O atraso é parecido entre as abordagens** (mediana de 3,0 a 4,6 h). O baseline ajustado alerta um pouco antes porque o limite dele é mais baixo, e paga isso em falsos positivos.
- **Todas as abordagens com motor por eventos cumprem o requisito 2** (cerca de um alerta por incidente encontrado).
- **Com 29 incidentes, as diferenças têm de ser lidas com o intervalo.** Nas mesmas reamostragens de cenários, a abordagem final ganha +0,13 de F1 sobre o baseline ajustado (intervalo de 90% de 0,00 a +0,26; melhor em 94% das reamostragens) e +0,13 sobre o PCA (−0,05 a +0,31; 88%). Nenhuma variante das seções 12 a 15 passou de 0,74 nessa probabilidade. Mesmo assim, o limite inferior do intervalo toca o zero: com 29 cenários, a conclusão honesta é "provavelmente melhor", não "comprovadamente melhor".

**Que métrica eu reportaria:** o F1 oficial, acompanhado do F1 fora da amostra (seção 16), de falsos positivos por ativo por mês e do atraso mediano. Essas quatro respondem às perguntas de quem vai receber os alertas: quanto confiar no número, quanto ruído esperar e quanto tempo antes da falha o aviso chega.

## Conclusões

**Placar** (baseline original como veio; ajustado e PCA nas janelas corrigidas; abordagem final nas janelas do YAML, como o `main.py`):

| | TP | FP | FN | Precisão | Recall | F1 |
|---|---|---|---|---|---|---|
| Baseline original | 5 | 14 | 24 | 0,26 | 0,17 | 0,21 |
| Baseline ajustado | 16 | 12 | 13 | 0,57 | 0,55 | 0,56 |
| PCA (T² + SPE) | 18 | 17 | 11 | 0,51 | 0,62 | 0,56 |
| **Nível sustentado (final)** | **17** | **3** | **12** | **0,85** | **0,59** | **0,69** |
| Final, fora da amostra (um cenário de fora) | 14 | 5 | 15 | 0,74 | 0,48 | 0,58 |

Nenhuma das três abordagens novas gera alerta nos sete cenários sem incidente; o baseline original gera um em quatro deles.

**A abordagem final é a que o `main.py` roda** (seção 16). Ela mantém o que as seções 4 a 15 mostraram que funciona (o nível da vibração, só com a máquina rodando, nas normas da velocidade e da aceleração, só para cima, e um motor de alertas por eventos) e troca as duas peças que limitavam o ajustado e o PCA:

- **O z-score vira uma razão ao topo do normal.** Um limite em desvios-padrão muda de significado de um ativo para outro (seção 13); 1,5× o percentil 99 do treino quer dizer a mesma coisa em todos.
- **A confirmação sai do motor e vai para o modelo.** O nível sustentado exige cerca de 1 h de evidência nas últimas 3 h, o motor alerta na primeira janela anormal e mede o tempo pelos timestamps.

O resultado são 3 falsos positivos em vez de 12 ou 17, para praticamente o mesmo recall, e uma melhor posição em todas as métricas complementares da seção 17. O custo é a sensibilidade a falhas de um eixo só, onde o PCA continua muito melhor (seção 16, teste de injeção).

As conclusões a seguir comparam o baseline ajustado e o PCA, a análise que levou à abordagem final.

**Poucas mudanças na regra original bastam para igualar o PCA na métrica.** As mudanças são o motor de alertas por eventos, só desvios para cima, a norma da aceleração ao lado da norma da velocidade, e só leituras com a máquina rodando com limite de 4,5 desvios-padrão. As duas abordagens empatam no F1: o baseline ajustado é mais preciso (12 falsos positivos contra 17) e o PCA encontra dois incidentes a mais. Elas concordam em normal/anormal em 85% das janelas.

**A métrica tem pouco poder para separar abordagens.** Com 29 cenários, um ou dois alertas movem o F1 em 0,02–0,04. Reamostrando os cenários, o F1 do baseline ajustado fica entre 0,44 e 0,68 em 90% das vezes, e a diferença entre o PCA e ele fica entre −0,15 e +0,15 (seção 15). Diferenças abaixo de uns 0,05 são empate, e isso vale para tudo o que foi testado nas seções 10 a 15.

**Onde o PCA ainda leva vantagem: falhas que não movem a máquina inteira.**
- **Um canal.** Uma norma dilui uma falha num único eixo. No teste de injeção, um canal multiplicado por 2 é sinalizado em 69% dos casos pelo PCA e em 14% pelo baseline ajustado (81% contra 38% com ×3). Quando sinalizam, os dois colocam o canal injetado em primeiro lugar em cerca de 92% das vezes.
- **Os seis canais.** Com ×1,5 o PCA sinaliza 56% dos casos e o baseline ajustado 15%. Com ×3 os dois sinalizam quase todos (93% e 96%).

**Onde o baseline ajustado leva vantagem: simplicidade.** Duas médias e dois desvios por ativo, menos falsos positivos e uma regra que um operador lê direto no gráfico de monitoramento da seção 5.

**O pipeline perde dados depois de um buraco.** `_windowing_ts` para na primeira janela vazia, então os cenários 5, 27 e 28 perdem as últimas 120 a 266 horas e um incidente cada. Janelas de 12 h com sobreposição de 11 h, pontuando só as últimas 2 h, evitam o problema sem mexer no código fixo. Isso acrescenta dois incidentes tanto ao baseline ajustado quanto ao PCA.

**A lógica de alerta pesa tanto quanto o detector.**
- **Regra de re-alerta.** Re-alertar quando o nível dobra é a melhor regra testada: F1 0,56, contra 0,48 sem re-alerta, 0,50 com CUSUM e 0,49 com a tendência. É também a única regra que dá o número desejado de re-alertas nos quatro perfis de manual.
- **Confirmação.** Tirar a confirmação (`onset_windows = 1`) derruba o F1 do PCA para 0,30 (seção 10). No baseline ajustado, confirmar em 2 janelas dobra os falsos positivos (12 → 24), e confirmar em 4 ou 6 perde os incidentes curtos (seção 15).
- **Recuperação.** Fechar os eventos após três janelas normais derruba o F1 do PCA para 0,32. Esperar 48 janelas em vez de 24 tira 2 falsos positivos do baseline ajustado (16/10/13) mas custa um incidente ao PCA (17/17/12), então 24 continua sendo o valor que serve às duas abordagens.

**O limite de 4,5 desvios-padrão é adequado; o que limita o resultado é ele ser fixo** (seção 13).
- É o melhor ponto da grade de limite × fração da janela, num platô (k = 4,5 e k = 5 dão o mesmo placar), e é escolhido em 26 das 29 dobras quando um cenário fica de fora.
- O mesmo k não significa a mesma coisa em todo ativo (a maior leitura do treino fica de 1,3 a 43,8 desvios da média), mas limites calculados por ativo a partir do treino ficam todos piores (0,38–0,54).
- Um limite por cenário, escolhido com os rótulos, chegaria a 20/1/9 (F1 0,80), com k de 2 a 20. O espaço para melhorar existe e está em adaptar o baseline a cada ativo ao longo do tempo, porque o nível normal da predição já não é o do treino em vários cenários.

**O que não ajudou** (seções 10 a 15).
- **Features extras no PCA** (pico, dispersão, razão vel/acc), no mesmo PCA ou como modelos separados (seção 11).
- **Estatísticas da janela na regra do baseline** (seção 12). Média, RMS e pico repetem o nível (no máximo 0,52 sozinhas); fator de crista, coeficiente de variação, curtose e assimetria não têm sinal (F1 até 0,12, e zero incidentes para curtose e assimetria). Somadas à regra, nenhuma passa de 0,58. Os dados são RMS a cada 10 min, sem forma de onda, então fator de crista e curtose do sinal não podem ser calculados.
- **Outras colunas julgadas** (seção 12): seis canais no lugar das duas normas, amplitude em log, mediana e MAD.
- **Limiar dinâmico de janela curta** (seção 14). Média móvel + desvio móvel de 24 h dá 0,26–0,34 e de 72 h dá 0,37–0,47: a falha entra na referência e o limiar a absorve. Mediana e MAD móveis são piores (0,26–0,43), e o baseline adaptativo com congelamento não passa de 0,58.
- **Regra em duas escalas** (2 h mais um trecho de 6 h ou 12 h): 0,52–0,56 (seção 15).
- **Outros intervalos pontuados ou passo de 30 min** (seção 10).

**O que parece promissor, sem estar comprovado** (seções 14 e 15). Os ganhos são de até +0,04 de F1 e a probabilidade de cada candidata ser melhor que o baseline ajustado não passa de 0,74.
- **Referência móvel de 14 dias com piso no desvio do treino:** 18/13/11, F1 0,60. Encontra 7b, 8 e 17, de cenários em que o nível normal se afastou do treino, e é a que ataca a causa apontada na seção 13. O intervalo do ganho é o mais largo (−0,06 a +0,15) e o modelo passaria a guardar histórico entre chamadas de `predict`.
- **Exigir concordância entre o baseline ajustado e o PCA:** 16/9/13, F1 0,59. Três falsos positivos a menos para o mesmo número de incidentes.
- **Um motor de alertas por sinal** (baseline ajustado e PCA, cada um com o próprio evento): 23/27/6, recall 0,79, F1 0,58. É o caminho para subir o recall, se o custo em alertas for aceitável para o produto.

**O que ainda está errado** (`a`/`b` = primeiro/segundo incidente do cenário).
- **Um incidente não pode ser encontrado:** o 7a, cujo rótulo termina antes de começar. O recall máximo é 28/29.
- **Incidentes sem desvio para cima acima do limite:** 11, 13 e 14 (onde a vibração *cai*) nas duas abordagens, mais o 27a no baseline ajustado e o 4a e o 29a no PCA.
- **O alerta sai antes do rótulo.** No baseline ajustado, cinco incidentes (4a, 8, 27b, 28b e 29b) começam com um evento já aberto e a severidade não dobra dentro da janela rotulada; cada um custa um falso positivo e um incidente perdido ao mesmo tempo. No PCA acontece o mesmo com 6b, 21 e 26.
- **Sinal fraco ou intermitente:** 6a, 7b e 17 no baseline ajustado têm menos de três janelas anormais seguidas. No PCA, o incidente do cenário 2 dura 2,5 h, pouco para a confirmação de três janelas, e o 27b fica nas últimas 10 h dos dados.
- **Falsos positivos:** 7 dos 12 do baseline ajustado caem a menos de ~28 h de uma janela rotulada, em desvios que começam antes do rótulo ou duram mais que ele; no PCA é cerca de metade. O resto é isolado, incluindo o baseline deslocado do cenário 10.
- **O limite da métrica.** No baseline ajustado, com 1 rótulo inválido, 4 incidentes sem desvio e 5 que começam depois de o alerta já ter saído, um detector de nível com um único estado de evento fica perto de 19 incidentes encontrados; separar os eventos por sinal leva a 23. Daí em diante o que separa as abordagens é quantos falsos positivos cada uma paga.

**Ressalvas.**
- **Os números são otimistas.** Todos os limites e hiperparâmetros foram escolhidos nos mesmos 29 cenários em que são pontuados. Escolhendo o limite com um cenário de fora, o F1 do baseline ajustado cai de 0,56 para 0,47 (seção 13). O limite de 4,5 fica num platô e é estável entre as dobras, o que o torna a mais confiável das escolhas.
- **O que os dados não permitem testar:** features do sinal bruto (fator de crista, curtose, bandas de frequência) e variáveis de contexto de operação, como carga e rotação, que explicariam parte das mudanças de nível que hoje viram falso positivo.

**O que foi para `classes/`.**
1. A abordagem final, como padrão: `classes/anomaly_model.py`, `classes/alert_engine.py` e os YAMLs `model_hyperparams`, `alert_hyperparams` e `pipeline_hyperparams` (janelas de 12 h com passo de 30 min).
2. O PCA (`classes/pca_model.py`, com o recorte do trecho final em `PcaAnomalyModel.predict`) e o motor que conta janelas (`classes/window_alert_engine.py`), como alternativas usadas por este notebook.
3. Ainda a revalidar com mais dados: a referência móvel de 14 dias com piso, a concordância entre dois detectores, e uma severidade por eixo atrás de um limite mais alto, para recuperar as falhas de um canal só sem trazer de volta os falsos positivos.